# Multi-Horizon Quantile Forecasting Notebook

Roadmap reference: **5-day hybrid daily-path forecast**.

This notebook trains the Dense, LSTM, and Transformer checkpoints consumed by `main.py`.

Key contracts:

- **Target.** The model predicts five forward closes using log-return labels `y_logret_h1 ... y_logret_h5`. A target row `t` is anchored to `Close[t-1]`, so `h=1` predicts `Close[t]` from an input window ending at `t-1`, and `h=5` predicts `Close[t+4]`.
- **Output head.** Each model emits **3 quantiles x 5 horizons = 15 scalars**, reshaped to `(batch, 3, 5)`.
- **No leakage.** Predictor windows are always rows `[t-SEQ_LEN, t-1]`; row `t` and all forward-label columns are target-only and excluded from features. Rolling indicators are shifted historically by `compute_indicators`.
- **Hybrid inference.** Live inference uses a short recursive one-step rollout for all five forecast columns. Actual closes are used when available during rollout; otherwise the previous predicted close is used. Missing future Open falls back to the previous close.
- **Compatibility.** The saved metadata records the five horizons, quantiles, feature columns, sequence length, model capacity, scheduled sampling settings, and forecasting strategy expected by `main.py`.


In [10]:
# ============================================================================
# Imports and Configuration
# ============================================================================
from pathlib import Path
from contextlib import nullcontext
from datetime import datetime, timedelta
import gc
import json
import math
import os
import random
import sys
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

try:
    import optuna
except ImportError:
    optuna = None

PROJECT_ROOT = Path("/Users/hrishavmajumder/Documents/Stock_Market")

# .env support (P0 requirement: split dates come from environment variables).
# This fallback matters on kernels without python-dotenv installed.
def _load_project_env_file(path):
    if not path.exists():
        return False
    loaded_any = False
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value
            loaded_any = True
    return loaded_any

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
    load_dotenv(Path.cwd() / ".env")
except ImportError:
    loaded = _load_project_env_file(PROJECT_ROOT / ".env") or _load_project_env_file(Path.cwd() / ".env")
    if loaded:
        print("python-dotenv not installed; loaded split variables from .env fallback.")
    else:
        print("python-dotenv not installed; relying on shell environment variables.")

warnings.filterwarnings("ignore")

SEED = 42

# ---------------------------------------------------------------------------
# Modelling config
# ---------------------------------------------------------------------------
SEQ_LEN = 20
BATCH_SIZE = 256
EPOCHS = 500
PATIENCE = 20
LEARNING_RATE = 1e-3

# Multi-horizon quantile config
FORWARD_HORIZONS = tuple(range(1, 6))   # h=1..5, anchored to the previous close
HORIZONS = list(FORWARD_HORIZONS)
QUANTILES = [0.10, 0.50, 0.90]         # q10 / q50 / q90
N_HORIZONS = len(HORIZONS)
N_QUANTILES = len(QUANTILES)
N_OUTPUTS = N_HORIZONS * N_QUANTILES   # 15
EMBARGO_DAYS = max(HORIZONS)

def _env_bool(name, default):
    raw = os.getenv(name)
    if raw is None or str(raw).strip() == "":
        return bool(default)
    return str(raw).strip().lower() in {"1", "true", "yes", "y", "on"}

# Hybrid inference uses recursive T+1..T+3 and direct multi-horizon T+4..T+5.
RECURSIVE_TRAIN_HORIZONS = 3
SCHEDULED_SAMPLING_START_PROB = 0.00
SCHEDULED_SAMPLING_END_PROB = 0.15
SCHEDULED_SAMPLING_WARMUP_EPOCHS = max(1, EPOCHS // 10)
SCHEDULED_SAMPLING_AUX_LOSS_WEIGHT = 0.25
SCHEDULED_SAMPLING_RETURN_CLIP = (-0.08, 0.08)
ROLLING_STABILIZATION_WINDOW = 20
USE_VOLATILITY_NORMALIZATION = False
SCHEDULED_SAMPLING_CONFIG = {
    "enabled": True,
    "recursive_horizons": RECURSIVE_TRAIN_HORIZONS,
    "start_probability": SCHEDULED_SAMPLING_START_PROB,
    "end_probability": SCHEDULED_SAMPLING_END_PROB,
    "warmup_epochs": SCHEDULED_SAMPLING_WARMUP_EPOCHS,
    "aux_loss_weight": SCHEDULED_SAMPLING_AUX_LOSS_WEIGHT,
    "return_clip": SCHEDULED_SAMPLING_RETURN_CLIP,
    "seed": SEED + 10_000,
}

# Optuna is intentionally notebook-native: no external orchestration files.
RUN_OPTUNA = _env_bool("RUN_OPTUNA", True)
OPTUNA_N_TRIALS = int(os.getenv("OPTUNA_N_TRIALS", "20"))
OPTUNA_TIMEOUT_SECONDS = int(os.getenv("OPTUNA_TIMEOUT_SECONDS", "3600"))
OPTUNA_TRIAL_EPOCHS = int(os.getenv("OPTUNA_TRIAL_EPOCHS", "35"))
OPTUNA_TRIAL_PATIENCE = int(os.getenv("OPTUNA_TRIAL_PATIENCE", "6"))
OPTUNA_VALIDATION_WINDOWS = int(os.getenv("OPTUNA_VALIDATION_WINDOWS", "3"))
OPTUNA_MIN_VALIDATION_POINTS = int(os.getenv("OPTUNA_MIN_VALIDATION_POINTS", "25"))
OPTUNA_PRUNER_WARMUP_STEPS = int(os.getenv("OPTUNA_PRUNER_WARMUP_STEPS", "8"))
OPTUNA_SCORE_WEIGHTS = {"rmse": 0.4, "mae": 0.3, "directional_accuracy": 0.3}

# Training window is anchored on TRAIN_END and extends 10 years back.
TRAIN_WINDOW_YEARS = 10

# Labels are forward log-returns of Close. They are TARGETS, NEVER features.
# Row t labels are anchored to Close[t-1], matching main.py's live forecast path.
FORWARD_LABEL_PREFIX = "y_logret_h"
LABEL_COLS = [f"{FORWARD_LABEL_PREFIX}{h}" for h in HORIZONS]
HAS_LABELS_COL = "has_labels"
TARGET_BASE_COL = "Close"      # used for inverse-transform back to price

# ---------------------------------------------------------------------------
# Date-based split configuration (read from .env / environment, parsed here)
# ---------------------------------------------------------------------------
def _env_value(name):
    for candidate in (name, name.lower()):
        raw = os.getenv(candidate)
        if raw is not None and str(raw).strip() != "":
            return str(raw).strip()
    return None

def _parse_env_date(name):
    raw = _env_value(name)
    if raw is None:
        raise RuntimeError(
            f"Environment variable {name} is required. "
            f"Add it to your .env (format: YYYY-MM-DD)."
        )
    return pd.Timestamp(raw).normalize()

TRAIN_END       = _parse_env_date("TRAIN_END")
TEST_END        = _parse_env_date("TEST_END")
BACK_TEST_START = _parse_env_date("BACK_TEST_START")
BACK_TEST_END   = _parse_env_date("BACK_TEST_END")

TRAIN_START = TRAIN_END - pd.DateOffset(years=TRAIN_WINDOW_YEARS)
TEST_START  = TRAIN_END + pd.Timedelta(days=1)  # (TRAIN_END, TEST_END]

if not (TRAIN_START < TRAIN_END < TEST_END):
    raise ValueError(
        f"Bad split: TRAIN_START={TRAIN_START.date()}, "
        f"TRAIN_END={TRAIN_END.date()}, TEST_END={TEST_END.date()}"
    )
if not (BACK_TEST_START <= BACK_TEST_END):
    raise ValueError(
        f"Bad backtest split: BACK_TEST_START={BACK_TEST_START.date()} > "
        f"BACK_TEST_END={BACK_TEST_END.date()}"
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Configuration loaded")
print(f"  TRAIN window     : {TRAIN_START.date()} -> {TRAIN_END.date()}")
print(f"  TEST  window     : {TEST_START.date()} -> {TEST_END.date()}")
print(f"  BACKTEST window  : {BACK_TEST_START.date()} -> {BACK_TEST_END.date()}")
print(f"  Embargo          : {EMBARGO_DAYS} trading rows")
print(f"  Horizons         : {HORIZONS[0]}..{HORIZONS[-1]} ({N_HORIZONS})")
print(f"  Quantiles        : {QUANTILES}")
print(f"  Output head size : {N_OUTPUTS}")
print(f"  Scheduled sampling: T+1..T+{RECURSIVE_TRAIN_HORIZONS}, p={SCHEDULED_SAMPLING_START_PROB:.2f}->{SCHEDULED_SAMPLING_END_PROB:.2f}")
print(f"  Optuna enabled    : {RUN_OPTUNA and optuna is not None} (trials={OPTUNA_N_TRIALS}, timeout={OPTUNA_TIMEOUT_SECONDS}s)")


python-dotenv not installed; relying on shell environment variables.
Configuration loaded
  TRAIN window     : 2013-12-31 -> 2023-12-31
  TEST  window     : 2024-01-01 -> 2025-06-30
  BACKTEST window  : 2025-07-01 -> 2025-12-31
  Embargo          : 5 trading rows
  Horizons         : 1..5 (5)
  Quantiles        : [0.1, 0.5, 0.9]
  Output head size : 15
  Scheduled sampling: T+1..T+3, p=0.00->0.15
  Optuna enabled    : True (trials=20, timeout=3600s)


In [11]:
# ============================================================================
# Device Setup
# ============================================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

USE_AMP = device.type == "cuda"
PIN_MEMORY = device.type == "cuda"
NUM_WORKERS = 0

if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"Device: {device} ({torch.cuda.get_device_name(0)})")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif device.type == "mps":
    print("Device: mps")
else:
    print("Device: cpu")

def autocast_context():
    if USE_AMP:
        return torch.cuda.amp.autocast()
    return nullcontext()

print(f"AMP enabled: {USE_AMP}")


Device: mps
AMP enabled: False


In [12]:
# ============================================================================
# Data Loading
# ============================================================================
PROJECT_ROOT = Path("/Users/hrishavmajumder/Documents/Stock_Market")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Feature_Engineering import (
    compute_indicators,
    compute_forward_log_returns,
    forward_label_columns,
    HAS_LABELS_COL as FE_HAS_LABELS_COL,
)

# Cross-check: the label-column contract must match between this notebook
# and Feature_Engineering. If a future edit changes the names, fail loudly here.
_expected_labels = forward_label_columns(tuple(HORIZONS))
if _expected_labels != LABEL_COLS:
    raise RuntimeError(
        f"Label column contract mismatch.\n"
        f"Notebook expects: {LABEL_COLS}\n"
        f"Feature_Engineering returns: {_expected_labels}"
    )
if FE_HAS_LABELS_COL != HAS_LABELS_COL:
    raise RuntimeError(
        f"has_labels column name mismatch: {FE_HAS_LABELS_COL!r} vs {HAS_LABELS_COL!r}"
    )

def find_excel_file():
    candidates = [
        PROJECT_ROOT / "Data" / "nse_stock_data.xlsx",
        PROJECT_ROOT / "Data" / "DATA_Stock.xlsx",
        PROJECT_ROOT / "nse_stock_data.xlsx",
        PROJECT_ROOT / "DATA_Stock.xlsx",
        Path.cwd() / "Data" / "nse_stock_data.xlsx",
        Path.cwd() / "Data" / "DATA_Stock.xlsx",
        Path.cwd() / "nse_stock_data.xlsx",
        Path.cwd() / "DATA_Stock.xlsx",
        Path("/kaggle/input/datasets/hrishavmajumder12/nse-stock-data-xlsx/nse_stock_data.xlsx"),
    ]
    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        excel_files = sorted(kaggle_input.rglob("*.xlsx"))
        if excel_files:
            return excel_files[0]
    searched = "\n".join(str(c) for c in candidates)
    raise FileNotFoundError(f"No .xlsx data file found. Searched:\n{searched}")

DATA_PATH = find_excel_file()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

xls = pd.ExcelFile(DATA_PATH)
print(f"Loaded workbook: {DATA_PATH}")
print(f"Sheets found: {len(xls.sheet_names)}")
print(f"First sheets: {xls.sheet_names[:10]}")


Loaded workbook: /Users/hrishavmajumder/Documents/Stock_Market/Data/nse_stock_data.xlsx
Sheets found: 49
First sheets: ['RELIANCE', 'TCS', 'INFY', 'HDFCBANK', 'ICICIBANK', 'SBIN', 'LT', 'ITC', 'HINDUNILVR', 'AXISBANK']


In [13]:
# ============================================================================
# Stock Normalization, Feature Alignment, and Forward-Label Attachment
# ============================================================================
PRICE_PREFIXES = {
    "open_": "Open", "high_": "High", "low_": "Low",
    "close_": "Close", "volume_": "Volume",
}
EXACT_PRICE_COLUMNS = {
    "open": "Open", "high": "High", "low": "Low",
    "close": "Close", "volume": "Volume",
}

def canonical_column_name(column):
    name = str(column).strip()
    lower = name.lower()
    if lower in EXACT_PRICE_COLUMNS:
        return EXACT_PRICE_COLUMNS[lower]
    for prefix, canonical in PRICE_PREFIXES.items():
        if lower.startswith(prefix):
            return canonical
    return name

def detect_date_column(df):
    cands = [c for c in df.columns if "date" in str(c).lower() or "time" in str(c).lower()]
    return cands[0] if cands else None

def is_forward_label_column(column):
    text = str(column).strip()
    prefix = FORWARD_LABEL_PREFIX
    return text.startswith(prefix) and text[len(prefix):].isdigit()

def compute_forecast_log_returns(close, horizons=FORWARD_HORIZONS):
    """
    Row t labels predict closes starting at t from data available through t-1.

    y_h(t) = log(Close[t + h - 1] / Close[t - 1]) for h in 1..5.
    """
    close = pd.to_numeric(close, errors="coerce")
    safe_close = close.where(close > 0)
    log_close = np.log(safe_close)
    anchor = log_close.shift(1)
    labels = {}
    for h in horizons:
        labels[f"{FORWARD_LABEL_PREFIX}{h}"] = log_close.shift(-(int(h) - 1)) - anchor
    return labels

def normalize_stock_sheet(df, symbol):
    df = df.copy()
    df.columns = [canonical_column_name(col) for col in df.columns]
    df = df.loc[:, ~pd.Index(df.columns).duplicated()].copy()

    date_col = detect_date_column(df)
    if date_col is not None:
        parsed = pd.to_datetime(df[date_col], errors="coerce")
        if parsed.notna().any():
            df[date_col] = parsed
            df = df.sort_values(date_col).reset_index(drop=True)
        else:
            raise ValueError(f"{symbol}: date column '{date_col}' has no parseable values")
    else:
        raise ValueError(f"{symbol}: no date column detected — required for date-based splits")

    drop_cols = [c for c in df.columns if "date_str" in str(c).lower() or "date_string" in str(c).lower()]
    if drop_cols:
        df = df.drop(columns=drop_cols, errors="ignore")

    for col in df.columns:
        if col == date_col:
            continue
        conv = pd.to_numeric(df[col], errors="coerce")
        if conv.notna().sum() > 0:
            df[col] = conv

    if TARGET_BASE_COL not in df.columns:
        raise ValueError(f"{symbol}: missing required column '{TARGET_BASE_COL}' after normalization")

    df[TARGET_BASE_COL] = pd.to_numeric(df[TARGET_BASE_COL], errors="coerce")
    df = df.dropna(subset=[TARGET_BASE_COL]).reset_index(drop=True)

    # Own the forward-label horizon contract in this notebook. Feature_Engineering
    # may attach default labels too, but this training run uses only h=1..5.
    forward_labels = compute_forecast_log_returns(df[TARGET_BASE_COL], horizons=FORWARD_HORIZONS)
    for label_name, label_series in forward_labels.items():
        df[label_name] = label_series.values
    df[HAS_LABELS_COL] = df[LABEL_COLS].notna().all(axis=1)

    # Remove legacy/default forward labels that Feature_Engineering may have
    # attached. All forward labels are targets and must never enter features.
    extra_forward_labels = [c for c in df.columns if is_forward_label_column(c) and c not in LABEL_COLS]
    if extra_forward_labels:
        df = df.drop(columns=extra_forward_labels, errors="ignore")

    # Numeric feature pool: every numeric column EXCEPT the close price (used as
    # the price anchor for inverse-transform) and all forward labels (targets).
    forbidden = {TARGET_BASE_COL, HAS_LABELS_COL, *LABEL_COLS}
    numeric_features = [
        c for c in df.select_dtypes(include=[np.number]).columns
        if c not in forbidden and not is_forward_label_column(c)
    ]
    if not numeric_features:
        raise ValueError(f"{symbol}: no numeric feature columns found")

    return df, numeric_features, date_col

stock_frames = {}
stock_feature_sets = {}
stock_date_cols = {}
skipped_stocks = {}

for sheet_name in tqdm(xls.sheet_names, desc="Normalizing sheets"):
    try:
        raw_df = pd.read_excel(xls, sheet_name=sheet_name)
        # compute_indicators builds historical-only shifted indicators;
        # normalize_stock_sheet rewrites forward labels for FORWARD_HORIZONS.
        engineered_df = compute_indicators(raw_df, detect_date_column(raw_df))
        normalized_df, numeric_features, date_col = normalize_stock_sheet(engineered_df, sheet_name)

        if HAS_LABELS_COL not in normalized_df.columns:
            raise ValueError(f"{sheet_name}: has_labels column missing — forward labels not attached")
        missing_label_cols = [c for c in LABEL_COLS if c not in normalized_df.columns]
        if missing_label_cols:
            raise ValueError(f"{sheet_name}: missing label columns {missing_label_cols[:3]}...")

        stock_frames[sheet_name] = normalized_df
        stock_feature_sets[sheet_name] = set(numeric_features)
        stock_date_cols[sheet_name] = date_col
        n_labeled = int(normalized_df[HAS_LABELS_COL].sum())
        print(f"{sheet_name}: rows={len(normalized_df)}, "
              f"features={len(numeric_features)}, has_labels rows={n_labeled}")
    except Exception as exc:
        skipped_stocks[sheet_name] = str(exc)
        print(f"Skipping {sheet_name}: {exc}")

if not stock_frames:
    raise RuntimeError("No valid stock sheets were loaded")

common_features = sorted(set.intersection(*stock_feature_sets.values()))
if not common_features:
    raise RuntimeError("No common numeric features found across valid stocks")

# Hard-stop feature contract: features must NOT include the close target or
# any forward label. This is the central no-leakage guarantee.
_forbidden_in_features = {TARGET_BASE_COL, HAS_LABELS_COL, *LABEL_COLS}
_leaked = sorted(
    c for c in common_features
    if c in _forbidden_in_features or is_forward_label_column(c)
)
if _leaked:
    raise RuntimeError(f"Leaked target/label columns in feature set: {_leaked}")

print(f"\nValid stocks: {len(stock_frames)}")
print(f"Skipped stocks: {len(skipped_stocks)}")
print(f"Common feature count: {len(common_features)}")
print(f"Common features: {common_features}")


Normalizing sheets:   0%|          | 0/49 [00:00<?, ?it/s]

RELIANCE: rows=2794, features=29, has_labels rows=2789
TCS: rows=2794, features=29, has_labels rows=2789
INFY: rows=2794, features=29, has_labels rows=2789
HDFCBANK: rows=2794, features=29, has_labels rows=2789
ICICIBANK: rows=2794, features=29, has_labels rows=2789
SBIN: rows=2794, features=29, has_labels rows=2789
LT: rows=2794, features=29, has_labels rows=2789
ITC: rows=2794, features=29, has_labels rows=2789
HINDUNILVR: rows=2794, features=29, has_labels rows=2789
AXISBANK: rows=2794, features=29, has_labels rows=2789
KOTAKBANK: rows=2794, features=29, has_labels rows=2789
BAJFINANCE: rows=2794, features=29, has_labels rows=2789
ASIANPAINT: rows=2794, features=29, has_labels rows=2789
BHARTIARTL: rows=2794, features=29, has_labels rows=2789
MARUTI: rows=2794, features=29, has_labels rows=2789
SUNPHARMA: rows=2794, features=29, has_labels rows=2789
ULTRACEMCO: rows=2794, features=29, has_labels rows=2789
NTPC: rows=2794, features=29, has_labels rows=2789
POWERGRID: rows=2794, featu

In [14]:
# ============================================================================
# Date-based Splits and Sequence Building (NO LEAKAGE)
# ============================================================================
# Leakage contract (do not relax):
#   1. A prediction anchored at row index t uses ONLY rows [t-SEQ_LEN, t-1].
#      Row t (today's Open / High / Low / Close / Volume / indicators) is
#      never read.
#   2. The feature scaler is fit on finite TRAIN-window feature rows only and
#      re-used frozen for test and backtest.
#   3. Forward labels are computed in Feature_Engineering, never derived from
#      the feature window inside the model code path.
#   4. Training labels must end inside the train window. The in-train validation
#      tail is separated from fit rows by a max-horizon trading-row embargo.
# ============================================================================

def split_indices_by_date(date_series, window_start, window_end):
    """Inclusive on both ends. Returns sorted integer positions."""
    ds = pd.to_datetime(date_series)
    mask = (ds >= window_start) & (ds <= window_end)
    return np.flatnonzero(mask.values)

def split_indices_open_left(date_series, window_start, window_end):
    """Half-open on the left: (window_start, window_end]. Used for TEST so
    that the day right after TRAIN_END is the first test day."""
    ds = pd.to_datetime(date_series)
    mask = (ds > window_start) & (ds <= window_end)
    return np.flatnonzero(mask.values)

def horizon_realisation_mask(df, target_indices, date_col, horizons, split_end):
    """Return mask[i, h] = True if y_h for target i is finite and inside split_end."""
    dates = pd.to_datetime(df[date_col]).reset_index(drop=True)
    label_values = df[LABEL_COLS].to_numpy(dtype=np.float32)
    target_indices = np.asarray(target_indices, dtype=np.int64)
    mask = np.zeros((len(target_indices), len(horizons)), dtype=bool)
    for row_i, t in enumerate(target_indices):
        for h_i, h in enumerate(horizons):
            future_idx = int(t) + int(h) - 1
            if future_idx >= len(df):
                continue
            if dates.iloc[future_idx] > split_end:
                continue
            if not np.isfinite(label_values[int(t), h_i]):
                continue
            mask[row_i, h_i] = True
    return mask

def build_sequences_for_target_indices(
    df,
    feature_columns,
    label_cols,
    target_indices,
    seq_len,
    feature_scaler,
    *,
    label_valid_mask=None,
    require_full_labels,
):
    """
    Construct (X, Y, valid_target_indices, valid_label_mask) for target_indices.

    X[i] is the scaled window of rows [t-seq_len, t-1] for t = target_indices[i].
    Y[i] is row-t forward-label vector y_h1..y_h5. For inference splits, Y can
    contain NaN, but valid_label_mask marks which horizons are eligible for metrics.
    Labels are anchored to Close[t-1], so horizon 1 realises at row t.

    NEVER reads row t for features. NEVER calls scaler.fit. Pure transform.
    """
    if feature_scaler is None or not hasattr(feature_scaler, "data_min_"):
        raise RuntimeError("feature_scaler must be fitted before sequence building")
    if TARGET_BASE_COL in feature_columns:
        raise RuntimeError(f"Feature contract violated: {TARGET_BASE_COL} present in features")
    for lc in label_cols:
        if lc in feature_columns:
            raise RuntimeError(f"Feature contract violated: label {lc} present in features")
    leaked_label_features = [col for col in feature_columns if is_forward_label_column(col)]
    if leaked_label_features:
        raise RuntimeError(f"Feature contract violated: forward labels present in features: {leaked_label_features}")

    target_indices = np.asarray(target_indices, dtype=np.int64)
    if label_valid_mask is None:
        label_valid_mask = np.isfinite(df.iloc[target_indices][label_cols].to_numpy(dtype=np.float32))
    else:
        label_valid_mask = np.asarray(label_valid_mask, dtype=bool)
        if label_valid_mask.shape != (len(target_indices), len(label_cols)):
            raise ValueError(
                f"label_valid_mask shape mismatch: {label_valid_mask.shape} vs "
                f"{(len(target_indices), len(label_cols))}"
            )

    feat_matrix = df[feature_columns].to_numpy(dtype=np.float32)
    scaled = feature_scaler.transform(feat_matrix).astype(np.float32)
    Y_full = df[label_cols].to_numpy(dtype=np.float32)

    X_list, Y_list, idx_list, mask_list = [], [], [], []
    dropped_nonfinite_window = 0
    dropped_label = 0
    for src_pos, t in enumerate(target_indices):
        t = int(t)
        if t < seq_len:
            continue
        y = Y_full[t]
        horizon_mask = label_valid_mask[src_pos] & np.isfinite(y)
        if require_full_labels and not horizon_mask.all():
            dropped_label += 1
            continue
        window = scaled[t - seq_len:t]  # rows [t-seq_len .. t-1] — t excluded
        if window.shape != (seq_len, len(feature_columns)):
            continue
        if not np.isfinite(window).all():
            dropped_nonfinite_window += 1
            continue
        last_window_row = t - 1
        if last_window_row >= t:
            raise RuntimeError("Sequence builder leaked future row")
        X_list.append(window)
        Y_list.append(y)
        idx_list.append(t)
        mask_list.append(horizon_mask)

    if not X_list:
        X = np.empty((0, seq_len, len(feature_columns)), dtype=np.float32)
        Y = np.empty((0, len(label_cols)), dtype=np.float32)
        idx = np.empty((0,), dtype=np.int64)
        valid_mask = np.empty((0, len(label_cols)), dtype=bool)
    else:
        X = np.stack(X_list).astype(np.float32)
        Y = np.stack(Y_list).astype(np.float32)
        idx = np.asarray(idx_list, dtype=np.int64)
        valid_mask = np.stack(mask_list).astype(bool)
    return X, Y, idx, valid_mask, {
        "dropped_nonfinite_window": int(dropped_nonfinite_window),
        "dropped_label": int(dropped_label),
    }

def split_train_fit_validation_indices(eligible_train_pos, val_frac=0.10, embargo=EMBARGO_DAYS):
    eligible_train_pos = np.asarray(eligible_train_pos, dtype=np.int64)
    if len(eligible_train_pos) <= embargo + 2:
        return eligible_train_pos, np.empty((0,), dtype=np.int64), np.empty((0,), dtype=np.int64)
    val_count = max(1, int(math.ceil(len(eligible_train_pos) * val_frac)))
    val_start = len(eligible_train_pos) - val_count
    fit_end = max(0, val_start - embargo)
    fit_pos = eligible_train_pos[:fit_end]
    embargo_pos = eligible_train_pos[fit_end:val_start]
    val_pos = eligible_train_pos[val_start:]
    return fit_pos, val_pos, embargo_pos

def assert_finite_numpy(name, arr):
    arr = np.asarray(arr)
    if not np.isfinite(arr).all():
        bad = int((~np.isfinite(arr)).sum())
        raise FloatingPointError(f"{name} contains {bad} non-finite values")

# ---------------------------------------------------------------------------
# Iterate stocks, fit per-stock feature scaler on TRAIN, then build sequences
# for train / validation / test / backtest using the SAME frozen scaler.
# ---------------------------------------------------------------------------
per_stock_artifacts = {}
sequence_failures = {}
sequence_filter_stats = {}

train_X_parts, train_Y_parts, train_sym_parts, train_idx_parts, train_mask_parts = [], [], [], [], []
inval_X_parts, inval_Y_parts, inval_sym_parts, inval_idx_parts, inval_mask_parts = [], [], [], [], []
test_X_parts,  test_Y_parts,  test_sym_parts,  test_idx_parts,  test_mask_parts  = [], [], [], [], []
back_X_parts,  back_Y_parts,  back_sym_parts,  back_idx_parts,  back_mask_parts  = [], [], [], [], []

for symbol, df in tqdm(stock_frames.items(), desc="Building sequences"):
    try:
        date_col = stock_date_cols[symbol]
        date_series = df[date_col]

        train_pos = split_indices_by_date(date_series, TRAIN_START, TRAIN_END)
        test_pos  = split_indices_open_left(date_series, TRAIN_END, TEST_END)
        back_pos  = split_indices_by_date(date_series, BACK_TEST_START, BACK_TEST_END)

        if len(train_pos) <= SEQ_LEN:
            raise ValueError(f"{symbol}: only {len(train_pos)} train rows (need > {SEQ_LEN})")

        feature_values = df[common_features].to_numpy(dtype=np.float32)
        finite_feature_rows = np.isfinite(feature_values).all(axis=1)
        finite_close_rows = np.isfinite(df[TARGET_BASE_COL].to_numpy(dtype=np.float32))
        train_rows_for_scaler = train_pos[finite_feature_rows[train_pos] & finite_close_rows[train_pos]]
        if len(train_rows_for_scaler) <= SEQ_LEN:
            raise ValueError(
                f"{symbol}: only {len(train_rows_for_scaler)} finite train rows for scalers"
            )

        # Fit scalers on finite TRAIN-window rows only — never on test/backtest.
        feature_scaler = MinMaxScaler()
        feature_scaler.fit(df.iloc[train_rows_for_scaler][common_features].to_numpy(dtype=np.float32))
        target_scaler = MinMaxScaler()
        target_scaler.fit(df.iloc[train_rows_for_scaler][[TARGET_BASE_COL]].to_numpy(dtype=np.float32))

        train_label_mask_all = horizon_realisation_mask(df, train_pos, date_col, HORIZONS, TRAIN_END)
        eligible_train_pos = train_pos[train_label_mask_all.all(axis=1)]
        if len(eligible_train_pos) <= SEQ_LEN + EMBARGO_DAYS:
            raise ValueError(
                f"{symbol}: only {len(eligible_train_pos)} leakage-free train anchors"
            )
        fit_pos, val_pos, embargo_pos = split_train_fit_validation_indices(
            eligible_train_pos, val_frac=0.10, embargo=EMBARGO_DAYS
        )
        if len(fit_pos) == 0 or len(val_pos) == 0:
            raise ValueError(f"{symbol}: empty fit/validation split after embargo")

        fit_label_mask = horizon_realisation_mask(df, fit_pos, date_col, HORIZONS, TRAIN_END)
        val_label_mask = horizon_realisation_mask(df, val_pos, date_col, HORIZONS, TRAIN_END)
        test_label_mask_all = horizon_realisation_mask(df, test_pos, date_col, HORIZONS, TEST_END)
        back_label_mask_all = horizon_realisation_mask(df, back_pos, date_col, HORIZONS, BACK_TEST_END)

        X_tr, Y_tr, idx_tr, mask_tr, stats_tr = build_sequences_for_target_indices(
            df, common_features, LABEL_COLS, fit_pos, SEQ_LEN,
            feature_scaler, label_valid_mask=fit_label_mask, require_full_labels=True,
        )
        X_va, Y_va, idx_va, mask_va, stats_va = build_sequences_for_target_indices(
            df, common_features, LABEL_COLS, val_pos, SEQ_LEN,
            feature_scaler, label_valid_mask=val_label_mask, require_full_labels=True,
        )
        X_te, Y_te, idx_te, mask_te, stats_te = build_sequences_for_target_indices(
            df, common_features, LABEL_COLS, test_pos, SEQ_LEN,
            feature_scaler, label_valid_mask=test_label_mask_all, require_full_labels=False,
        )
        X_bk, Y_bk, idx_bk, mask_bk, stats_bk = build_sequences_for_target_indices(
            df, common_features, LABEL_COLS, back_pos, SEQ_LEN,
            feature_scaler, label_valid_mask=back_label_mask_all, require_full_labels=False,
        )

        if len(X_tr) == 0 or len(X_va) == 0 or len(X_te) == 0:
            raise ValueError(
                f"{symbol}: empty after sequence build "
                f"(train={len(X_tr)}, val={len(X_va)}, test={len(X_te)}, backtest={len(X_bk)})"
            )

        assert_finite_numpy(f"{symbol} X_train", X_tr)
        assert_finite_numpy(f"{symbol} Y_train", Y_tr)
        assert_finite_numpy(f"{symbol} X_validation", X_va)
        assert_finite_numpy(f"{symbol} Y_validation", Y_va)
        assert_finite_numpy(f"{symbol} X_test", X_te)
        if len(X_bk) > 0:
            assert_finite_numpy(f"{symbol} X_backtest", X_bk)

        train_X_parts.append(X_tr); train_Y_parts.append(Y_tr); train_mask_parts.append(mask_tr)
        train_sym_parts.append(np.full(len(X_tr), symbol, dtype=object)); train_idx_parts.append(idx_tr)

        inval_X_parts.append(X_va); inval_Y_parts.append(Y_va); inval_mask_parts.append(mask_va)
        inval_sym_parts.append(np.full(len(X_va), symbol, dtype=object)); inval_idx_parts.append(idx_va)

        test_X_parts.append(X_te); test_Y_parts.append(Y_te); test_mask_parts.append(mask_te)
        test_sym_parts.append(np.full(len(X_te), symbol, dtype=object)); test_idx_parts.append(idx_te)

        if len(X_bk) > 0:
            back_X_parts.append(X_bk); back_Y_parts.append(Y_bk); back_mask_parts.append(mask_bk)
            back_sym_parts.append(np.full(len(X_bk), symbol, dtype=object)); back_idx_parts.append(idx_bk)

        stats = {
            "fit": stats_tr,
            "validation": stats_va,
            "test": stats_te,
            "backtest": stats_bk,
            "embargo_rows": int(len(embargo_pos)),
        }
        sequence_filter_stats[symbol] = stats
        per_stock_artifacts[symbol] = {
            "source_frame": df.reset_index(drop=True).copy(),
            "date_col": date_col,
            "feature_scaler": feature_scaler,
            "target_scaler": target_scaler,
            "train_target_indices": idx_tr,
            "validation_target_indices": idx_va,
            "test_target_indices": idx_te,
            "backtest_target_indices": idx_bk,
            "test_label_mask": mask_te,
            "backtest_label_mask": mask_bk,
        }
        print(
            f"{symbol}: train={len(X_tr)}, val={len(X_va)}, test={len(X_te)}, "
            f"backtest={len(X_bk)}, embargo={len(embargo_pos)}, window=({SEQ_LEN}, {len(common_features)})"
        )
    except Exception as exc:
        sequence_failures[symbol] = str(exc)
        print(f"Skipping {symbol}: {exc}")

def _concat(parts_X, parts_Y, parts_sym, parts_idx, parts_mask, name):
    if not parts_X:
        raise RuntimeError(f"No {name} sequences across any stock")
    X = np.concatenate(parts_X, axis=0).astype(np.float32)
    Y = np.concatenate(parts_Y, axis=0).astype(np.float32)
    sym = np.concatenate(parts_sym, axis=0)
    idx = np.concatenate(parts_idx, axis=0).astype(np.int64)
    mask = np.concatenate(parts_mask, axis=0).astype(bool)
    return X, Y, sym, idx, mask

X_train, Y_train, train_symbols, train_target_indices, train_label_mask = _concat(
    train_X_parts, train_Y_parts, train_sym_parts, train_idx_parts, train_mask_parts, "train"
)
X_inval, Y_inval, inval_symbols, inval_target_indices, inval_label_mask = _concat(
    inval_X_parts, inval_Y_parts, inval_sym_parts, inval_idx_parts, inval_mask_parts, "validation"
)
X_test, Y_test, test_symbols, test_target_indices, test_label_mask = _concat(
    test_X_parts, test_Y_parts, test_sym_parts, test_idx_parts, test_mask_parts, "test"
)
if back_X_parts:
    X_back, Y_back, back_symbols, back_target_indices, back_label_mask = _concat(
        back_X_parts, back_Y_parts, back_sym_parts, back_idx_parts, back_mask_parts, "backtest"
    )
else:
    X_back = np.empty((0, SEQ_LEN, len(common_features)), dtype=np.float32)
    Y_back = np.empty((0, N_HORIZONS), dtype=np.float32)
    back_symbols = np.empty((0,), dtype=object)
    back_target_indices = np.empty((0,), dtype=np.int64)
    back_label_mask = np.empty((0, N_HORIZONS), dtype=bool)
    print("WARNING: backtest split produced zero rows — check BACK_TEST_START/END")

for name, X, Y, mask, require_all_y in [
    ("train", X_train, Y_train, train_label_mask, True),
    ("validation", X_inval, Y_inval, inval_label_mask, True),
    ("test", X_test, Y_test, test_label_mask, False),
    ("backtest", X_back, Y_back, back_label_mask, False),
]:
    assert_finite_numpy(f"{name} X", X)
    if require_all_y:
        assert_finite_numpy(f"{name} Y", Y)
    if len(mask) and not mask.any():
        raise RuntimeError(f"{name} has no realised horizons available for metrics")

actual_seq_len = X_train.shape[1]
actual_feature_count = X_train.shape[2]
actual_input_size = actual_seq_len * actual_feature_count

print("\nGlobal tensor shapes")
print(f"  Train fit X: {X_train.shape}, Y: {Y_train.shape}")
print(f"  In-train validation X: {X_inval.shape}, Y: {Y_inval.shape}")
print(f"  Test     X: {X_test.shape}, Y: {Y_test.shape}, metric horizons={int(test_label_mask.sum())}")
print(f"  Backtest X: {X_back.shape}, Y: {Y_back.shape}, metric horizons={int(back_label_mask.sum())}")
print(f"  input_size (Dense) = {actual_seq_len} * {actual_feature_count} = {actual_input_size}")
print(f"  output head size   = {N_OUTPUTS} ({N_QUANTILES} quantiles x {N_HORIZONS} horizons)")
print(f"  Sequence failures  : {len(sequence_failures)}")


Building sequences:   0%|          | 0/49 [00:00<?, ?it/s]

RELIANCE: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
TCS: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
INFY: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
HDFCBANK: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
ICICIBANK: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
SBIN: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
LT: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
ITC: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
HINDUNILVR: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
AXISBANK: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
KOTAKBANK: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
BAJFINANCE: train=1920, val=222, test=369, backtest=126, embargo=5, window=(20, 29)
ASIANPAINT: train=1920, val=222, t

In [15]:
# ============================================================================
# Dataset and DataLoaders (multi-output Y of shape (samples, n_horizons))
# ============================================================================
class StockMultiHorizonDataset(Dataset):
    def __init__(self, X, Y, expected_seq_len, expected_feature_count, expected_horizons, *, require_finite_y=True):
        X = np.asarray(X, dtype=np.float32)
        Y = np.asarray(Y, dtype=np.float32)
        if X.ndim != 3:
            raise ValueError(f"X must be 3D, got {X.shape}")
        if Y.ndim != 2:
            raise ValueError(f"Y must be 2D (samples, n_horizons), got {Y.shape}")
        if len(X) != len(Y):
            raise ValueError(f"X/Y sample mismatch: {len(X)} vs {len(Y)}")
        if X.shape[1] != expected_seq_len or X.shape[2] != expected_feature_count:
            raise ValueError(
                f"X shape mismatch: expected (*, {expected_seq_len}, "
                f"{expected_feature_count}), got {X.shape}"
            )
        if Y.shape[1] != expected_horizons:
            raise ValueError(f"Y shape mismatch: expected (*, {expected_horizons}), got {Y.shape}")
        if not np.isfinite(X).all():
            raise FloatingPointError(f"X contains {int((~np.isfinite(X)).sum())} non-finite values")
        if require_finite_y and not np.isfinite(Y).all():
            raise FloatingPointError(f"Y contains {int((~np.isfinite(Y)).sum())} non-finite values")

        self.X = torch.as_tensor(X, dtype=torch.float32).contiguous()
        self.Y = torch.as_tensor(Y, dtype=torch.float32).contiguous()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

def make_loader(dataset, batch_size, num_workers, pin_memory, shuffle=False):
    if shuffle:
        raise ValueError("Shuffling disabled for chronological time-series training")
    kwargs = dict(batch_size=batch_size, shuffle=False, num_workers=num_workers,
                  pin_memory=pin_memory, drop_last=False)
    if num_workers > 0:
        kwargs.update({"prefetch_factor": 2, "persistent_workers": True})
    return DataLoader(dataset, **kwargs)

train_dataset = StockMultiHorizonDataset(
    X_train, Y_train, actual_seq_len, actual_feature_count, N_HORIZONS, require_finite_y=True
)
inval_dataset = StockMultiHorizonDataset(
    X_inval, Y_inval, actual_seq_len, actual_feature_count, N_HORIZONS, require_finite_y=True
)
test_dataset  = StockMultiHorizonDataset(
    X_test, Y_test, actual_seq_len, actual_feature_count, N_HORIZONS, require_finite_y=False
)
if len(X_back) > 0:
    back_dataset = StockMultiHorizonDataset(
        X_back, Y_back, actual_seq_len, actual_feature_count, N_HORIZONS, require_finite_y=False
    )
else:
    back_dataset = None

train_loader = make_loader(train_dataset, BATCH_SIZE, NUM_WORKERS, PIN_MEMORY)
inval_loader = make_loader(inval_dataset, BATCH_SIZE, NUM_WORKERS, PIN_MEMORY)
test_loader  = make_loader(test_dataset,  BATCH_SIZE, NUM_WORKERS, PIN_MEMORY)
back_loader  = make_loader(back_dataset,  BATCH_SIZE, NUM_WORKERS, PIN_MEMORY) if back_dataset is not None else None

for name, loader in [("train", train_loader), ("validation", inval_loader), ("test", test_loader)] + (
    [("backtest", back_loader)] if back_loader is not None else []
):
    Xb, Yb = next(iter(loader))
    print(f"{name}: X batch {tuple(Xb.shape)}, Y batch {tuple(Yb.shape)}")
    assert Xb.dim() == 3 and Yb.dim() == 2
    assert Xb.shape[1] == actual_seq_len
    assert Xb.shape[2] == actual_feature_count
    assert Yb.shape[1] == N_HORIZONS
    if not torch.isfinite(Xb).all():
        raise FloatingPointError(f"{name} X batch contains non-finite values")


train: X batch (256, 20, 29), Y batch (256, 5)
validation: X batch (256, 20, 29), Y batch (256, 5)
test: X batch (256, 20, 29), Y batch (256, 5)
backtest: X batch (256, 20, 29), Y batch (256, 5)


In [16]:
# ============================================================================
# Multi-Horizon Quantile Model Architectures
# ============================================================================
# Each model emits N_OUTPUTS = N_QUANTILES * N_HORIZONS scalars per sample,
# reshaped to (batch, N_QUANTILES, N_HORIZONS).
# ============================================================================

class DenseQuantileModel(nn.Module):
    def __init__(self, input_size, n_quantiles, n_horizons, hidden_size=128, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.n_quantiles = n_quantiles
        self.n_horizons = n_horizons
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_size // 2, n_quantiles * n_horizons)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout1(x)
        x = self.relu(self.fc2(x))
        x = self.dropout2(x)
        out = self.fc3(x)
        return out.view(-1, self.n_quantiles, self.n_horizons)

class LSTMQuantileModel(nn.Module):
    def __init__(self, input_features, n_quantiles, n_horizons,
                 hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_features
        self.n_quantiles = n_quantiles
        self.n_horizons = n_horizons
        self.lstm = nn.LSTM(
            input_size=input_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, n_quantiles * n_horizons),
        )

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        out = self.head(hidden[-1])
        return out.view(-1, self.n_quantiles, self.n_horizons)

class TransformerQuantileModel(nn.Module):
    def __init__(self, input_features, n_quantiles, n_horizons,
                 hidden_size=128, num_heads=4, num_layers=2, dropout=0.3,
                 feedforward_dim=None, max_seq_len=512):
        super().__init__()
        if hidden_size % num_heads != 0:
            raise ValueError(f"hidden_size ({hidden_size}) must be divisible by num_heads ({num_heads})")
        self.input_size = input_features
        self.n_quantiles = n_quantiles
        self.n_horizons = n_horizons
        feedforward_dim = int(feedforward_dim or hidden_size * 4)
        self.feedforward_dim = feedforward_dim
        self.input_projection = nn.Linear(input_features, hidden_size)
        self.position_embedding = nn.Parameter(torch.zeros(1, max_seq_len, hidden_size))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, n_quantiles * n_horizons),
        )

    def forward(self, x):
        seq_len = x.shape[1]
        if seq_len > self.position_embedding.shape[1]:
            raise ValueError(
                f"Sequence length {seq_len} exceeds max_seq_len "
                f"{self.position_embedding.shape[1]}"
            )
        x = self.input_projection(x)
        x = x + self.position_embedding[:, :seq_len, :]
        x = self.encoder(x)
        out = self.head(x[:, -1, :])
        return out.view(-1, self.n_quantiles, self.n_horizons)

print(f"Dense input_size from X: {actual_input_size}")
print(f"LSTM/Transformer input_features from X.shape[2]: {actual_feature_count}")
print(f"Each model output shape: (batch, {N_QUANTILES}, {N_HORIZONS})")


Dense input_size from X: 580
LSTM/Transformer input_features from X.shape[2]: 29
Each model output shape: (batch, 3, 5)


In [17]:
# ============================================================================
# Pinball Loss + Training/Eval Utilities (all dependencies passed as args)
# ============================================================================

def assert_finite_tensor(name, tensor):
    if not torch.isfinite(tensor).all():
        bad = int((~torch.isfinite(tensor)).sum().detach().cpu().item())
        raise FloatingPointError(f"{name} contains {bad} non-finite values")

def pinball_loss(predictions, targets, quantiles_tensor, *, check_tensors=True):
    """
    Multi-horizon multi-quantile pinball (quantile) loss.

    predictions: (batch, n_quantiles, n_horizons)  -- model output
    targets:     (batch, n_horizons)               -- forward log-returns
    quantiles_tensor: (n_quantiles,)               -- e.g. [0.10, 0.50, 0.90]

    Returns the mean pinball loss across batch x quantiles x horizons.
    """
    if predictions.dim() != 3:
        raise ValueError(f"predictions must be (B,Q,H), got {tuple(predictions.shape)}")
    if targets.dim() != 2:
        raise ValueError(f"targets must be (B,H), got {tuple(targets.shape)}")
    if check_tensors:
        assert_finite_tensor("pinball predictions", predictions)
        assert_finite_tensor("pinball targets", targets)
    target_expanded = targets.unsqueeze(1)  # (B, 1, H)
    diff = target_expanded - predictions    # (B, Q, H)
    q = quantiles_tensor.view(1, -1, 1)     # (1, Q, 1)
    loss = torch.maximum(q * diff, (q - 1) * diff).mean()
    if check_tensors:
        assert_finite_tensor("pinball loss", loss)
    return loss

def sort_quantiles_non_crossing(predictions_np):
    """Post-hoc sort along the quantile axis so q10 <= q50 <= q90 per horizon."""
    predictions_np = np.asarray(predictions_np, dtype=np.float32)
    if not np.isfinite(predictions_np).all():
        raise FloatingPointError("Predictions contain non-finite values before quantile sorting")
    return np.sort(predictions_np, axis=1)

def log_first_batch_shape(model, dataloader, model_name, expected_input_size,
                           expected_feature_count):
    Xb, Yb = next(iter(dataloader))
    print(f"\n{model_name} runtime shape validation")
    print(f"  X batch shape: {tuple(Xb.shape)}")
    print(f"  Y batch shape: {tuple(Yb.shape)}")
    assert_finite_tensor(f"{model_name} first X batch", Xb)
    assert_finite_tensor(f"{model_name} first Y batch", Yb)
    if hasattr(model, "fc1"):
        actual = Xb.view(Xb.size(0), -1).shape[1]
        if actual != expected_input_size:
            raise ValueError(f"{model_name}: dense input mismatch {actual} vs {expected_input_size}")
    if Xb.shape[2] != expected_feature_count:
        raise ValueError(f"{model_name}: feature count mismatch {Xb.shape[2]} vs {expected_feature_count}")

def scheduled_sampling_probability(epoch, epochs, config):
    if not config or not bool(config.get("enabled", False)):
        return 0.0
    start_prob = float(config.get("start_probability", 0.0))
    end_prob = float(config.get("end_probability", 0.0))
    warmup_epochs = int(config.get("warmup_epochs", 0))
    if end_prob <= 0.0:
        return 0.0
    if epoch <= warmup_epochs:
        return max(0.0, min(1.0, start_prob))
    span = max(int(epochs) - warmup_epochs, 1)
    progress = min(max((int(epoch) - warmup_epochs) / span, 0.0), 1.0)
    return max(0.0, min(1.0, start_prob + (end_prob - start_prob) * progress))

def scheduled_sampling_feature_indices(feature_columns):
    price_like = {
        "Open", "High", "Low", "SMA_5", "SMA_20", "SMA_50",
        "EMA_12", "EMA_26", "EMA_50", "BB_Upper_20", "BB_Middle_20",
        "BB_Lower_20", "VWAP",
    }
    return [
        idx for idx, column in enumerate(feature_columns)
        if column in price_like or "Return_%" in str(column) or str(column) == "ROC_12"
    ]

def scheduled_sampling_rollout_batch(
    Xb,
    predictions,
    probability,
    feature_columns,
    quantile_idx,
    config,
    generator,
):
    """
    Build train-time recursive-style windows for the short horizon only.

    The Close target is intentionally not a model feature, so the injected future
    rows perturb only scaled close-derived feature proxies using clipped q50 model
    returns. No true future OHLCV or forward labels are read into the inputs.
    """
    if probability <= 0.0:
        return Xb
    batch_size = Xb.size(0)
    sample_mask = torch.rand(batch_size, generator=generator, device="cpu").to(Xb.device) < probability
    if not bool(sample_mask.any()):
        return Xb

    feature_indices = scheduled_sampling_feature_indices(feature_columns)
    volume_idx = feature_columns.index("Volume") if "Volume" in feature_columns else None
    if not feature_indices and volume_idx is None:
        return Xb

    recursive_horizons = min(int(config.get("recursive_horizons", 5)), predictions.shape[-1])
    low_clip, high_clip = config.get("return_clip", (-0.08, 0.08))
    q50_returns = predictions[:, quantile_idx, :recursive_horizons].detach()
    X_roll = Xb.clone()
    mask_view = sample_mask.view(-1, 1, 1)

    for horizon_idx in range(recursive_horizons):
        clipped_return = torch.clamp(q50_returns[:, horizon_idx], float(low_clip), float(high_clip))
        next_row = X_roll[:, -1, :].clone()
        if feature_indices:
            adjustment = clipped_return.view(-1, 1)
            next_row[:, feature_indices] = torch.clamp(next_row[:, feature_indices] + adjustment, 0.0, 1.0)
        if volume_idx is not None:
            recent_volume = X_roll[:, -min(5, X_roll.shape[1]):, volume_idx].mean(dim=1)
            next_row[:, volume_idx] = recent_volume
        candidate = torch.cat([X_roll[:, 1:, :], next_row.unsqueeze(1)], dim=1)
        X_roll = torch.where(mask_view, candidate, X_roll)

    assert_finite_tensor("scheduled sampling X", X_roll)
    return X_roll

def train_quantile_model(
    model, model_name, train_loader, val_loader, *,
    quantiles, device, epochs, learning_rate, patience,
    use_amp, pin_memory, expected_input_size, expected_feature_count,
    feature_columns, scheduled_sampling_config=None, weight_decay=1e-5,
    trial=None, trial_step_offset=0, verbose=True, check_tensors=True,
):
    """Train one model with pinball loss. All training-time dependencies are
    explicit arguments (no closure capture of notebook globals)."""
    model = model.to(device)
    if verbose:
        log_first_batch_shape(model, train_loader, model_name,
                               expected_input_size, expected_feature_count)

    quantiles_tensor = torch.tensor(quantiles, dtype=torch.float32, device=device)
    quantile_idx = quantiles.index(0.50) if 0.50 in quantiles else len(quantiles) // 2
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=float(weight_decay))
    amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    ss_generator = torch.Generator(device="cpu")
    ss_generator.manual_seed(int((scheduled_sampling_config or {}).get("seed", 0)))

    best_val = float("inf")
    best_state = None
    patience_counter = 0
    train_losses, val_losses = [], []

    def _ac():
        return torch.cuda.amp.autocast() if use_amp else nullcontext()

    epoch_iter = tqdm(range(1, epochs + 1), desc=f"Training {model_name}", disable=not verbose)
    for epoch in epoch_iter:
        model.train()
        total, n = 0.0, 0
        ss_prob = scheduled_sampling_probability(epoch, epochs, scheduled_sampling_config)
        ss_weight = float((scheduled_sampling_config or {}).get("aux_loss_weight", 0.0))
        for batch_i, (Xb, Yb) in enumerate(train_loader, start=1):
            Xb = Xb.to(device, non_blocking=pin_memory)
            Yb = Yb.to(device, non_blocking=pin_memory)
            if check_tensors:
                assert_finite_tensor(f"{model_name} train X batch {batch_i}", Xb)
                assert_finite_tensor(f"{model_name} train Y batch {batch_i}", Yb)
            optimizer.zero_grad(set_to_none=True)
            with _ac():
                preds = model(Xb)
                loss = pinball_loss(preds, Yb, quantiles_tensor, check_tensors=check_tensors)
                if ss_prob > 0.0 and ss_weight > 0.0:
                    X_ss = scheduled_sampling_rollout_batch(
                        Xb,
                        preds,
                        ss_prob,
                        feature_columns,
                        quantile_idx,
                        scheduled_sampling_config,
                        ss_generator,
                    )
                    if X_ss is not Xb:
                        ss_preds = model(X_ss)
                        loss = loss + ss_weight * pinball_loss(ss_preds, Yb, quantiles_tensor, check_tensors=check_tensors)
            if use_amp:
                amp_scaler.scale(loss).backward()
                amp_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                amp_scaler.step(optimizer)
                amp_scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            bs = Xb.size(0)
            total += float(loss.item()) * bs
            n += bs
        train_loss = total / max(n, 1)
        if not np.isfinite(train_loss):
            raise FloatingPointError(f"{model_name}: non-finite train loss at epoch {epoch}")

        val_loss = evaluate_pinball(model, val_loader, quantiles_tensor, device,
                                      use_amp, pin_memory, model_name=model_name, check_tensors=check_tensors)
        if not np.isfinite(val_loss):
            raise FloatingPointError(f"{model_name}: non-finite validation loss at epoch {epoch}")
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if trial is not None:
            trial.report(float(val_loss), int(trial_step_offset + epoch))
            if trial.should_prune():
                raise optuna.TrialPruned(f"{model_name} pruned at epoch {epoch}")
        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(f"{model_name} epoch {epoch:03d}: train={train_loss:.6f}, val={val_loss:.6f}, ss_p={ss_prob:.3f}")
        if patience_counter >= patience:
            if verbose:
                print(f"{model_name}: early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    for param_name, tensor in model.state_dict().items():
        if not torch.isfinite(tensor).all():
            raise FloatingPointError(f"{model_name} parameter {param_name} contains non-finite values")
    return model, train_losses, val_losses

def evaluate_pinball(model, loader, quantiles_tensor, device, use_amp, pin_memory, *, model_name="model", check_tensors=True):
    model.eval()
    total, n = 0.0, 0
    def _ac():
        return torch.cuda.amp.autocast() if use_amp else nullcontext()
    with torch.no_grad():
        for batch_i, (Xb, Yb) in enumerate(loader, start=1):
            Xb = Xb.to(device, non_blocking=pin_memory)
            Yb = Yb.to(device, non_blocking=pin_memory)
            if check_tensors:
                assert_finite_tensor(f"{model_name} eval X batch {batch_i}", Xb)
                assert_finite_tensor(f"{model_name} eval Y batch {batch_i}", Yb)
            with _ac():
                preds = model(Xb)
                loss = pinball_loss(preds, Yb, quantiles_tensor, check_tensors=check_tensors)
            bs = Xb.size(0)
            total += float(loss.item()) * bs
            n += bs
    return total / max(n, 1)


In [18]:
# ============================================================================
# Optuna Hyperparameter Optimization (walk-forward-safe, all 5 horizons)
# ============================================================================
# The study trains Dense/LSTM/Transformer together per trial and scores the
# ensemble over all validation rows, symbols, validation windows, and horizons.
# It never changes the feature/label leakage contract: sequences still end at
# t-1, forward labels stay target-only, and scalers are the frozen train scalers.
# ============================================================================

DEFAULT_OPTUNA_HPARAMS = {
    "sequence_length": int(actual_seq_len),
    "batch_size": int(BATCH_SIZE),
    "learning_rate": float(LEARNING_RATE),
    "weight_decay": 1e-5,
    "dropout": 0.30,
    "dense_width": 128,
    "lstm_hidden_size": 64,
    "lstm_layers": 2,
    "transformer_hidden_size": 128,
    "transformer_heads": 4,
    "transformer_layers": 2,
    "transformer_feedforward_multiplier": 4,
    "transformer_feedforward_dim": 512,
    "recursive_horizon_cutoff": 3,
    "direct_recursive_blend": 0.15,
    "return_clip_threshold": 0.08,
    "rolling_window_length": 20,
    "use_volatility_normalization": False,
    "scheduled_sampling_end_probability": SCHEDULED_SAMPLING_END_PROB,
    "scheduled_sampling_aux_weight": SCHEDULED_SAMPLING_AUX_LOSS_WEIGHT,
    "horizon_weight_tilt": 1.0,
    "ensemble_weights": {"Dense": 0.2, "LSTM": 0.3, "Transformer": 0.5},
}

OPTUNA_RESULTS = {
    "status": "disabled",
    "best_score": None,
    "best_trial_number": None,
    "best_params": DEFAULT_OPTUNA_HPARAMS.copy(),
    "horizon_metrics": [],
    "validation_methodology": "chronological train tail with embargo; validation scored in walk-forward date windows",
}
OPTUNA_SELECTED_HPARAMS = DEFAULT_OPTUNA_HPARAMS.copy()


def cleanup_trial_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def sequence_length_choices(max_seq_len):
    candidates = [10, 15, 20, 30, int(max_seq_len)]
    choices = sorted({int(c) for c in candidates if 5 <= int(c) <= int(max_seq_len)})
    return choices or [int(max_seq_len)]


def tail_sequences(X, seq_len):
    X = np.asarray(X, dtype=np.float32)
    seq_len = int(seq_len)
    if X.ndim != 3 or seq_len < 1 or seq_len > X.shape[1]:
        raise ValueError(f"invalid sequence length {seq_len} for X shape {X.shape}")
    return X[:, -seq_len:, :].astype(np.float32)


def build_hpo_datasets(seq_len, batch_size):
    seq_len = int(seq_len)
    batch_size = int(batch_size)
    X_tr = tail_sequences(X_train, seq_len)
    X_va = tail_sequences(X_inval, seq_len)
    train_ds = StockMultiHorizonDataset(
        X_tr, Y_train, seq_len, actual_feature_count, N_HORIZONS, require_finite_y=True
    )
    val_ds = StockMultiHorizonDataset(
        X_va, Y_inval, seq_len, actual_feature_count, N_HORIZONS, require_finite_y=True
    )
    return (
        X_tr,
        X_va,
        make_loader(train_ds, batch_size, NUM_WORKERS, PIN_MEMORY),
        make_loader(val_ds, batch_size, NUM_WORKERS, PIN_MEMORY),
    )


def validation_anchor_dates(symbols, target_indices, artifacts):
    values = []
    for symbol, target_idx in zip(symbols, target_indices):
        art = artifacts[str(symbol)]
        df = art["source_frame"].reset_index(drop=True)
        date_col = art["date_col"]
        values.append(pd.Timestamp(df.iloc[int(target_idx)][date_col]).normalize())
    return np.asarray(values, dtype="datetime64[ns]")


def build_validation_windows(anchor_dates, n_windows):
    anchor_dates = np.asarray(anchor_dates, dtype="datetime64[ns]")
    valid = ~pd.isna(anchor_dates)
    if valid.sum() == 0:
        return [np.ones(len(anchor_dates), dtype=bool)]
    order = np.argsort(anchor_dates[valid], kind="stable")
    valid_positions = np.flatnonzero(valid)[order]
    chunks = np.array_split(valid_positions, max(1, min(int(n_windows), len(valid_positions))))
    windows = []
    for chunk in chunks:
        if len(chunk) == 0:
            continue
        mask = np.zeros(len(anchor_dates), dtype=bool)
        mask[chunk] = True
        windows.append(mask)
    return windows or [np.ones(len(anchor_dates), dtype=bool)]


VALIDATION_ANCHOR_DATES = validation_anchor_dates(inval_symbols, inval_target_indices, per_stock_artifacts)
VALIDATION_WINDOWS = build_validation_windows(VALIDATION_ANCHOR_DATES, OPTUNA_VALIDATION_WINDOWS)


def normalized_model_weights(params):
    raw = {
        "Dense": max(0.0, float(params.get("dense_ensemble_weight", params.get("ensemble_weights", {}).get("Dense", 0.2)))),
        "LSTM": max(0.0, float(params.get("lstm_ensemble_weight", params.get("ensemble_weights", {}).get("LSTM", 0.3)))),
        "Transformer": max(0.0, float(params.get("transformer_ensemble_weight", params.get("ensemble_weights", {}).get("Transformer", 0.5)))),
    }
    total = sum(raw.values())
    if total <= 0:
        return {name: 1.0 / len(raw) for name in raw}
    return {name: value / total for name, value in raw.items()}


def build_models_from_hparams(params, seq_len):
    dense_width = int(params["dense_width"])
    lstm_hidden = int(params["lstm_hidden_size"])
    lstm_layers = int(params["lstm_layers"])
    transformer_hidden = int(params["transformer_hidden_size"])
    transformer_heads = int(params["transformer_heads"])
    transformer_layers = int(params["transformer_layers"])
    transformer_ff = int(params["transformer_feedforward_dim"])
    dropout = float(params["dropout"])
    if transformer_hidden % transformer_heads != 0:
        raise ValueError("Transformer hidden size must be divisible by heads")
    return {
        "Dense": DenseQuantileModel(
            input_size=int(seq_len) * actual_feature_count,
            n_quantiles=N_QUANTILES,
            n_horizons=N_HORIZONS,
            hidden_size=dense_width,
            dropout=dropout,
        ),
        "LSTM": LSTMQuantileModel(
            input_features=actual_feature_count,
            n_quantiles=N_QUANTILES,
            n_horizons=N_HORIZONS,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            dropout=dropout,
        ),
        "Transformer": TransformerQuantileModel(
            input_features=actual_feature_count,
            n_quantiles=N_QUANTILES,
            n_horizons=N_HORIZONS,
            hidden_size=transformer_hidden,
            num_heads=transformer_heads,
            num_layers=transformer_layers,
            dropout=dropout,
            feedforward_dim=transformer_ff,
            max_seq_len=max(512, int(seq_len)),
        ),
    }


def predict_ensemble_array(models, loader, weights, *, device, use_amp):
    used_weights = normalize_ensemble_weights(models.keys(), weights)
    per_model = {}
    actuals = None
    for name, model in models.items():
        preds, y = predict_quantile_model(model, loader, device=device, use_amp=use_amp, pin_memory=PIN_MEMORY)
        per_model[name] = preds
        actuals = y if actuals is None else actuals
    ensemble = np.zeros_like(next(iter(per_model.values())), dtype=np.float32)
    for name, preds in per_model.items():
        ensemble += used_weights[name] * preds
    return sort_quantiles_non_crossing(ensemble), actuals, used_weights


def stabilize_log_return_path(q50_BH, *, recursive_cutoff, blend, clip_threshold):
    direct = np.asarray(q50_BH, dtype=np.float64)
    if direct.ndim != 2 or direct.shape[1] != N_HORIZONS:
        raise ValueError(f"q50 path must be (*, {N_HORIZONS}), got {direct.shape}")
    increments = np.diff(np.concatenate([np.zeros((len(direct), 1)), direct], axis=1), axis=1)
    increments = np.clip(increments, -float(clip_threshold), float(clip_threshold))
    recursive_proxy = np.cumsum(increments, axis=1)
    hybrid = direct.copy()
    cutoff = max(1, min(int(recursive_cutoff), N_HORIZONS))
    hybrid[:, :cutoff] = recursive_proxy[:, :cutoff]
    if cutoff < N_HORIZONS:
        blend = float(np.clip(blend, 0.0, 1.0))
        hybrid[:, cutoff:] = (1.0 - blend) * direct[:, cutoff:] + blend * recursive_proxy[:, cutoff:]
    return hybrid.astype(np.float32)


def horizon_weights_from_tilt(tilt):
    tilt = float(tilt)
    if N_HORIZONS == 1:
        return np.ones(1, dtype=np.float64)
    weights = np.linspace(1.0, max(0.2, tilt), N_HORIZONS, dtype=np.float64)
    return weights / np.mean(weights)


def validation_score(actuals_BH, preds_BH, *, label_mask, windows, params):
    actuals = np.asarray(actuals_BH, dtype=np.float64)
    preds = np.asarray(preds_BH, dtype=np.float64)
    mask = np.asarray(label_mask, dtype=bool) & np.isfinite(actuals) & np.isfinite(preds)
    if int(mask.sum()) < OPTUNA_MIN_VALIDATION_POINTS:
        raise ValueError(f"too few validation points: {int(mask.sum())}")
    horizon_weights = horizon_weights_from_tilt(params.get("horizon_weight_tilt", 1.0))
    score_rows = []
    horizon_rows = []
    all_weighted_sq, all_weighted_abs = [], []
    all_dirs = []
    for h_i, horizon in enumerate(HORIZONS):
        h_mask = mask[:, h_i]
        if not h_mask.any():
            horizon_rows.append({"horizon": int(horizon), "rmse": None, "mae": None, "directional_accuracy": None, "n": 0})
            continue
        y = actuals[h_mask, h_i]
        p = preds[h_mask, h_i]
        direction = (np.sign(y) == np.sign(p)).astype(np.float64)
        rmse = float(np.sqrt(np.mean((y - p) ** 2)))
        mae = float(np.mean(np.abs(y - p)))
        horizon_rows.append({
            "horizon": int(horizon),
            "rmse": rmse,
            "mae": mae,
            "directional_accuracy": float(direction.mean()),
            "n": int(h_mask.sum()),
        })
        all_weighted_sq.append(((y - p) ** 2) * horizon_weights[h_i])
        all_weighted_abs.append(np.abs(y - p) * horizon_weights[h_i])
        all_dirs.append(direction)
    if not all_weighted_sq:
        raise ValueError("no finite validation horizons")
    flat_sq = np.concatenate(all_weighted_sq)
    flat_abs = np.concatenate(all_weighted_abs)
    flat_dirs = np.concatenate(all_dirs)
    rmse = float(np.sqrt(np.mean(flat_sq)))
    mae = float(np.mean(flat_abs))
    directional_accuracy = float(np.mean(flat_dirs))

    if bool(params.get("use_volatility_normalization", False)):
        scale_source = np.where(mask, np.abs(actuals), np.nan)
        normalizer = float(np.nanmedian(np.nanstd(scale_source, axis=1)))
    else:
        normalizer = float(np.nanmedian(np.abs(actuals[mask])))
    normalizer = max(normalizer, 1e-4)
    normalized_rmse = rmse / normalizer
    normalized_mae = mae / normalizer
    score = (
        OPTUNA_SCORE_WEIGHTS["rmse"] * normalized_rmse
        + OPTUNA_SCORE_WEIGHTS["mae"] * normalized_mae
        - OPTUNA_SCORE_WEIGHTS["directional_accuracy"] * directional_accuracy
    )

    for window_idx, window_mask in enumerate(windows, start=1):
        w_mask = mask & window_mask[:, None]
        if not w_mask.any():
            continue
        w_rmse = float(np.sqrt(np.mean((actuals[w_mask] - preds[w_mask]) ** 2)))
        w_mae = float(np.mean(np.abs(actuals[w_mask] - preds[w_mask])))
        w_dir = float(np.mean(np.sign(actuals[w_mask]) == np.sign(preds[w_mask])))
        score_rows.append({"window": window_idx, "rmse": w_rmse, "mae": w_mae, "directional_accuracy": w_dir, "n": int(w_mask.sum())})

    return float(score), {
        "rmse": rmse,
        "mae": mae,
        "normalized_rmse": normalized_rmse,
        "normalized_mae": normalized_mae,
        "directional_accuracy": directional_accuracy,
        "horizon_metrics": horizon_rows,
        "window_metrics": score_rows,
        "normalizer": normalizer,
    }


def make_trial_params(trial):
    seq_choices = sequence_length_choices(actual_seq_len)
    transformer_hidden = trial.suggest_categorical("transformer_hidden_size", [64, 128, 256])
    transformer_heads = trial.suggest_categorical("transformer_heads", [2, 4, 8])
    transformer_ff_multiplier = trial.suggest_int("transformer_feedforward_multiplier", 2, 6, step=2)
    params = {
        "sequence_length": trial.suggest_categorical("sequence_length", seq_choices),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-7, 1e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.05, 0.45),
        "dense_width": trial.suggest_categorical("dense_width", [64, 128, 256, 512]),
        "lstm_hidden_size": trial.suggest_categorical("lstm_hidden_size", [32, 64, 128, 256]),
        "lstm_layers": trial.suggest_int("lstm_layers", 1, 3),
        "transformer_hidden_size": transformer_hidden,
        "transformer_heads": transformer_heads,
        "transformer_layers": trial.suggest_int("transformer_layers", 1, 4),
        "transformer_feedforward_multiplier": transformer_ff_multiplier,
        "transformer_feedforward_dim": int(transformer_hidden) * int(transformer_ff_multiplier),
        "recursive_horizon_cutoff": trial.suggest_int("recursive_horizon_cutoff", 3, 3),
        "direct_recursive_blend": trial.suggest_float("direct_recursive_blend", 0.0, 0.40),
        "return_clip_threshold": trial.suggest_float("return_clip_threshold", 0.04, 0.12),
        "rolling_window_length": trial.suggest_int("rolling_window_length", 10, 40, step=5),
        "use_volatility_normalization": trial.suggest_categorical("use_volatility_normalization", [False, True]),
        "scheduled_sampling_end_probability": trial.suggest_float("scheduled_sampling_end_probability", 0.05, 0.30),
        "scheduled_sampling_aux_weight": trial.suggest_float("scheduled_sampling_aux_weight", 0.05, 0.40),
        "horizon_weight_tilt": trial.suggest_float("horizon_weight_tilt", 0.75, 1.50),
        "dense_ensemble_weight": trial.suggest_float("dense_ensemble_weight", 0.05, 1.0),
        "lstm_ensemble_weight": trial.suggest_float("lstm_ensemble_weight", 0.05, 1.0),
        "transformer_ensemble_weight": trial.suggest_float("transformer_ensemble_weight", 0.05, 1.0),
    }
    params["ensemble_weights"] = normalized_model_weights(params)
    return params


def scheduled_sampling_for_params(params, trial_number=0):
    cfg = dict(SCHEDULED_SAMPLING_CONFIG)
    clip = float(params.get("return_clip_threshold", 0.08))
    cfg.update({
        "recursive_horizons": int(params.get("recursive_horizon_cutoff", 3)),
        "end_probability": float(params.get("scheduled_sampling_end_probability", SCHEDULED_SAMPLING_END_PROB)),
        "aux_loss_weight": float(params.get("scheduled_sampling_aux_weight", SCHEDULED_SAMPLING_AUX_LOSS_WEIGHT)),
        "return_clip": (-clip, clip),
        "seed": int(SEED + 10_000 + trial_number),
    })
    return cfg


def optuna_objective(trial):
    params = make_trial_params(trial)
    cleanup_trial_memory()
    try:
        seq_len = int(params["sequence_length"])
        _, _, train_trial_loader, val_trial_loader = build_hpo_datasets(seq_len, params["batch_size"])
        trial_models = build_models_from_hparams(params, seq_len)
        trial_ss_cfg = scheduled_sampling_for_params(params, trial.number)
        for model_i, (model_name, model) in enumerate(trial_models.items()):
            trained_model, _, _ = train_quantile_model(
                model,
                model_name,
                train_trial_loader,
                val_trial_loader,
                quantiles=QUANTILES,
                device=device,
                epochs=OPTUNA_TRIAL_EPOCHS,
                learning_rate=params["learning_rate"],
                patience=OPTUNA_TRIAL_PATIENCE,
                use_amp=USE_AMP,
                pin_memory=PIN_MEMORY,
                expected_input_size=seq_len * actual_feature_count,
                expected_feature_count=actual_feature_count,
                feature_columns=common_features,
                scheduled_sampling_config=trial_ss_cfg,
                weight_decay=params["weight_decay"],
                trial=trial,
                trial_step_offset=model_i * OPTUNA_TRIAL_EPOCHS,
                verbose=False,
                check_tensors=False,
            )
            trial_models[model_name] = trained_model
        ensemble_BQH, actuals_BH, used_w = predict_ensemble_array(
            trial_models,
            val_trial_loader,
            params["ensemble_weights"],
            device=device,
            use_amp=USE_AMP,
        )
        q50_idx = QUANTILES.index(0.50) if 0.50 in QUANTILES else len(QUANTILES) // 2
        q50_path = ensemble_BQH[:, q50_idx, :]
        hybrid_q50 = stabilize_log_return_path(
            q50_path,
            recursive_cutoff=params["recursive_horizon_cutoff"],
            blend=params["direct_recursive_blend"],
            clip_threshold=params["return_clip_threshold"],
        )
        score, score_details = validation_score(
            actuals_BH,
            hybrid_q50,
            label_mask=inval_label_mask,
            windows=VALIDATION_WINDOWS,
            params=params,
        )
        if not np.isfinite(score):
            raise optuna.TrialPruned("non-finite validation score")
        trial.report(float(score), int(len(trial_models) * OPTUNA_TRIAL_EPOCHS + 1))
        if trial.should_prune():
            raise optuna.TrialPruned("pruned after full validation score")
        trial.set_user_attr("forecast_metrics", score_details)
        trial.set_user_attr("ensemble_weights", used_w)
        trial.set_user_attr("recursive_direct_mix", {
            "recursive_horizons": list(range(1, int(params["recursive_horizon_cutoff"]) + 1)),
            "direct_horizons": list(range(int(params["recursive_horizon_cutoff"]) + 1, N_HORIZONS + 1)),
            "direct_recursive_blend": float(params["direct_recursive_blend"]),
        })
        return score
    except optuna.TrialPruned:
        cleanup_trial_memory()
        raise
    except (FloatingPointError, ValueError, AssertionError) as exc:
        cleanup_trial_memory()
        trial.set_user_attr("failure", str(exc))
        raise optuna.TrialPruned(f"invalid trial: {exc}")
    except RuntimeError as exc:
        cleanup_trial_memory()
        if "out of memory" in str(exc).lower() or "cuda" in str(exc).lower():
            raise optuna.TrialPruned(f"GPU/runtime trial failure: {exc}")
        trial.set_user_attr("failure", str(exc))
        return float("inf")
    except KeyboardInterrupt as exc:
        cleanup_trial_memory()
        trial.set_user_attr("failure", "interrupted")
        raise optuna.TrialPruned("trial interrupted") from exc
    except Exception as exc:
        cleanup_trial_memory()
        trial.set_user_attr("failure", str(exc))
        return float("inf")
    finally:
        cleanup_trial_memory()


def run_optuna_study():
    if optuna is None:
        print("Optuna is not installed; using default hyperparameters.")
        return None
    if not RUN_OPTUNA:
        print("RUN_OPTUNA is disabled; using default hyperparameters.")
        return None
    if len(X_train) == 0 or len(X_inval) == 0:
        print("Insufficient train/validation tensors for Optuna; using defaults.")
        return None
    sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True, group=True)
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=max(3, min(8, OPTUNA_N_TRIALS // 4)),
        n_warmup_steps=OPTUNA_PRUNER_WARMUP_STEPS,
    )
    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
    study.optimize(
        optuna_objective,
        n_trials=OPTUNA_N_TRIALS,
        timeout=OPTUNA_TIMEOUT_SECONDS if OPTUNA_TIMEOUT_SECONDS > 0 else None,
        gc_after_trial=True,
        show_progress_bar=True,
        catch=(Exception,),
    )
    return study


optuna_study = run_optuna_study()
if optuna_study is not None and len(optuna_study.trials) > 0 and optuna_study.best_trial.value is not None and np.isfinite(optuna_study.best_trial.value):
    best_params = dict(DEFAULT_OPTUNA_HPARAMS)
    best_params.update(optuna_study.best_trial.params)
    best_params["ensemble_weights"] = normalized_model_weights(best_params)
    OPTUNA_SELECTED_HPARAMS = best_params
    forecast_metrics = optuna_study.best_trial.user_attrs.get("forecast_metrics", {})
    recursive_mix = optuna_study.best_trial.user_attrs.get("recursive_direct_mix", {})
    OPTUNA_RESULTS = {
        "status": "completed",
        "best_score": float(optuna_study.best_value),
        "best_trial_number": int(optuna_study.best_trial.number),
        "best_params": best_params,
        "horizon_metrics": forecast_metrics.get("horizon_metrics", []),
        "window_metrics": forecast_metrics.get("window_metrics", []),
        "ensemble_weights": best_params["ensemble_weights"],
        "recursive_direct_mix": recursive_mix,
        "validation_methodology": "chronological train tail with embargo; all validation points scored in walk-forward date windows",
    }
    print("\nOptuna best trial")
    print(f"  trial: {OPTUNA_RESULTS['best_trial_number']}")
    print(f"  score: {OPTUNA_RESULTS['best_score']:.6f}")
    print(f"  ensemble_weights: {OPTUNA_RESULTS['ensemble_weights']}")
    print(f"  recursive/direct: {OPTUNA_RESULTS['recursive_direct_mix']}")
    if OPTUNA_RESULTS["horizon_metrics"]:
        print(pd.DataFrame(OPTUNA_RESULTS["horizon_metrics"]).to_string(index=False))
else:
    print("Optuna did not produce a finite best trial; using default hyperparameters.")

if optuna_study is not None:
    try:
        optuna_history_df = optuna_study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
        print("\nOptuna trial summary:")
        print(optuna_history_df[["number", "value", "state"]].tail(10).to_string(index=False))
    except Exception as exc:
        print(f"Could not render Optuna trial summary: {exc}")


[I 2026-05-20 00:42:17,668] A new study created in memory with name: no-name-ed331479-6911-443f-8dd2-e8c599d43feb


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-05-20 00:51:01,994] Trial 0 finished with value: inf and parameters: {'transformer_hidden_size': 128, 'transformer_heads': 2, 'transformer_feedforward_multiplier': 2, 'sequence_length': 10, 'batch_size': 128, 'learning_rate': 3.357296705351791e-05, 'weight_decay': 5.337032762603952e-07, 'dropout': 0.12336180394137353, 'dense_width': 128, 'lstm_hidden_size': 32, 'lstm_layers': 2, 'transformer_layers': 4, 'recursive_horizon_cutoff': 3, 'direct_recursive_blend': 0.0798695128633439, 'return_clip_threshold': 0.08113875507308892, 'rolling_window_length': 30, 'use_volatility_normalization': True, 'scheduled_sampling_end_probability': 0.09263103092182289, 'scheduled_sampling_aux_weight': 0.07276805754484783, 'horizon_weight_tilt': 1.4616641529399999, 'dense_ensemble_weight': 0.9673504314208313, 'lstm_ensemble_weight': 0.817977480710638, 'transformer_ensemble_weight': 0.33938308071470213}. Best is trial 0 with value: inf.
[I 2026-05-20 00:55:13,822] Trial 1 finished with value: inf and 

In [19]:
# ============================================================================
# Train Dense, LSTM, Transformer (multi-horizon quantile heads)
# ============================================================================
# The final export path remains unchanged: all three model families are trained
# and saved under Dense.pt, LSTM.pt, and Transformer.pt. If Optuna completed,
# this cell consumes the selected hyperparameters; otherwise it falls back to
# the notebook defaults.
# ============================================================================

selected_hparams = dict(DEFAULT_OPTUNA_HPARAMS)
selected_hparams.update(OPTUNA_SELECTED_HPARAMS)
selected_hparams["transformer_feedforward_multiplier"] = int(selected_hparams.get("transformer_feedforward_multiplier", 4))
selected_hparams["transformer_feedforward_dim"] = int(selected_hparams["transformer_hidden_size"]) * int(selected_hparams["transformer_feedforward_multiplier"])
selected_hparams["ensemble_weights"] = normalized_model_weights(selected_hparams)

active_seq_len = int(selected_hparams["sequence_length"])
active_batch_size = int(selected_hparams["batch_size"])
active_learning_rate = float(selected_hparams["learning_rate"])
active_weight_decay = float(selected_hparams["weight_decay"])
active_dropout = float(selected_hparams["dropout"])
active_recursive_horizons = int(selected_hparams["recursive_horizon_cutoff"])
active_return_clip = float(selected_hparams["return_clip_threshold"])

X_train_model = tail_sequences(X_train, active_seq_len)
X_inval_model = tail_sequences(X_inval, active_seq_len)
X_test_model = tail_sequences(X_test, active_seq_len)
X_back_model = tail_sequences(X_back, active_seq_len) if len(X_back) > 0 else np.empty((0, active_seq_len, actual_feature_count), dtype=np.float32)

actual_seq_len = active_seq_len
actual_input_size = actual_seq_len * actual_feature_count
SEQ_LEN = active_seq_len
BATCH_SIZE = active_batch_size
RECURSIVE_TRAIN_HORIZONS = active_recursive_horizons
SCHEDULED_SAMPLING_RETURN_CLIP = (-active_return_clip, active_return_clip)
ROLLING_STABILIZATION_WINDOW = int(selected_hparams["rolling_window_length"])
USE_VOLATILITY_NORMALIZATION = bool(selected_hparams.get("use_volatility_normalization", False))
SCHEDULED_SAMPLING_CONFIG = scheduled_sampling_for_params(selected_hparams, trial_number=10_000)
ensemble_weights = selected_hparams["ensemble_weights"]

train_fit_dataset = StockMultiHorizonDataset(
    X_train_model, Y_train, actual_seq_len, actual_feature_count, N_HORIZONS, require_finite_y=True
)
inval_dataset = StockMultiHorizonDataset(
    X_inval_model, Y_inval, actual_seq_len, actual_feature_count, N_HORIZONS, require_finite_y=True
)
test_dataset = StockMultiHorizonDataset(
    X_test_model, Y_test, actual_seq_len, actual_feature_count, N_HORIZONS, require_finite_y=False
)
back_dataset = StockMultiHorizonDataset(
    X_back_model, Y_back, actual_seq_len, actual_feature_count, N_HORIZONS, require_finite_y=False
) if len(X_back_model) > 0 else None

train_fit_loader = make_loader(train_fit_dataset, BATCH_SIZE, NUM_WORKERS, PIN_MEMORY)
inval_loader     = make_loader(inval_dataset,     BATCH_SIZE, NUM_WORKERS, PIN_MEMORY)
test_loader      = make_loader(test_dataset,      BATCH_SIZE, NUM_WORKERS, PIN_MEMORY)
back_loader      = make_loader(back_dataset,      BATCH_SIZE, NUM_WORKERS, PIN_MEMORY) if back_dataset is not None else None

model_capacity = {
    "Dense": {
        "hidden_size": int(selected_hparams["dense_width"]),
        "dropout": active_dropout,
    },
    "LSTM": {
        "hidden_size": int(selected_hparams["lstm_hidden_size"]),
        "num_layers": int(selected_hparams["lstm_layers"]),
        "dropout": active_dropout,
    },
    "Transformer": {
        "hidden_size": int(selected_hparams["transformer_hidden_size"]),
        "num_heads": int(selected_hparams["transformer_heads"]),
        "num_layers": int(selected_hparams["transformer_layers"]),
        "feedforward_dim": int(selected_hparams["transformer_feedforward_dim"]),
        "dropout": active_dropout,
    },
}

models = {
    "Dense": DenseQuantileModel(
        input_size=actual_input_size,
        n_quantiles=N_QUANTILES,
        n_horizons=N_HORIZONS,
        hidden_size=model_capacity["Dense"]["hidden_size"],
        dropout=active_dropout,
    ),
    "LSTM": LSTMQuantileModel(
        input_features=actual_feature_count,
        n_quantiles=N_QUANTILES,
        n_horizons=N_HORIZONS,
        hidden_size=model_capacity["LSTM"]["hidden_size"],
        num_layers=model_capacity["LSTM"]["num_layers"],
        dropout=active_dropout,
    ),
    "Transformer": TransformerQuantileModel(
        input_features=actual_feature_count,
        n_quantiles=N_QUANTILES,
        n_horizons=N_HORIZONS,
        hidden_size=model_capacity["Transformer"]["hidden_size"],
        num_heads=model_capacity["Transformer"]["num_heads"],
        num_layers=model_capacity["Transformer"]["num_layers"],
        dropout=active_dropout,
        feedforward_dim=model_capacity["Transformer"]["feedforward_dim"],
        max_seq_len=max(512, actual_seq_len),
    ),
}

print("\nFinal training hyperparameters")
print(json.dumps({k: v for k, v in selected_hparams.items() if k != "ensemble_weights"}, indent=2, default=str))
print(f"Final ensemble weights: {ensemble_weights}")
print(f"Active sequence shape: (*, {actual_seq_len}, {actual_feature_count}); Dense input={actual_input_size}")

trained_models = {}
history = {}
model_failures = {}
for name, model in models.items():
    try:
        trained_model, tr_losses, va_losses = train_quantile_model(
            model, name, train_fit_loader, inval_loader,
            quantiles=QUANTILES, device=device, epochs=EPOCHS,
            learning_rate=active_learning_rate, patience=PATIENCE,
            use_amp=USE_AMP, pin_memory=PIN_MEMORY,
            expected_input_size=actual_input_size,
            expected_feature_count=actual_feature_count,
            feature_columns=common_features,
            scheduled_sampling_config=SCHEDULED_SAMPLING_CONFIG,
            weight_decay=active_weight_decay,
            verbose=True,
        )
        trained_models[name] = trained_model
        history[name] = {"train": tr_losses, "val": va_losses}
        cleanup_trial_memory()
    except Exception as exc:
        model_failures[name] = str(exc)
        cleanup_trial_memory()
        print(f"FAILED {name}: {exc}")

if not trained_models:
    raise RuntimeError(f"All models failed: {model_failures}")

print("\nTraining complete")
print(f"  Models trained: {list(trained_models.keys())}")
print(f"  Model failures: {model_failures}")



Final training hyperparameters
{
  "sequence_length": 20,
  "batch_size": 256,
  "learning_rate": 0.001,
  "weight_decay": 1e-05,
  "dropout": 0.3,
  "dense_width": 128,
  "lstm_hidden_size": 64,
  "lstm_layers": 2,
  "transformer_hidden_size": 128,
  "transformer_heads": 4,
  "transformer_layers": 2,
  "transformer_feedforward_multiplier": 4,
  "transformer_feedforward_dim": 512,
  "recursive_horizon_cutoff": 3,
  "direct_recursive_blend": 0.15,
  "return_clip_threshold": 0.08,
  "rolling_window_length": 20,
  "use_volatility_normalization": false,
  "scheduled_sampling_end_probability": 0.15,
  "scheduled_sampling_aux_weight": 0.25,
  "horizon_weight_tilt": 1.0
}
Final ensemble weights: {'Dense': 0.2, 'LSTM': 0.3, 'Transformer': 0.5}
Active sequence shape: (*, 20, 29); Dense input=580

Dense runtime shape validation
  X batch shape: (256, 20, 29)
  Y batch shape: (256, 5)


Training Dense:   0%|          | 0/500 [00:00<?, ?it/s]

Dense epoch 001: train=0.009314, val=0.006156, ss_p=0.000
Dense epoch 005: train=0.007871, val=0.006118, ss_p=0.000
Dense epoch 010: train=0.007861, val=0.006125, ss_p=0.000
Dense epoch 015: train=0.007862, val=0.006114, ss_p=0.000
Dense epoch 020: train=0.007860, val=0.006095, ss_p=0.000
Dense epoch 025: train=0.007853, val=0.006090, ss_p=0.000
Dense epoch 030: train=0.007852, val=0.006081, ss_p=0.000
Dense epoch 035: train=0.007851, val=0.006067, ss_p=0.000
Dense epoch 040: train=0.007843, val=0.006048, ss_p=0.000
Dense epoch 045: train=0.007844, val=0.006070, ss_p=0.000
Dense epoch 050: train=0.007841, val=0.006034, ss_p=0.000
Dense epoch 055: train=0.008563, val=0.006032, ss_p=0.002
Dense epoch 060: train=0.008971, val=0.006038, ss_p=0.003
Dense epoch 065: train=0.009378, val=0.006020, ss_p=0.005
Dense epoch 070: train=0.009508, val=0.006028, ss_p=0.007
Dense epoch 075: train=0.009565, val=0.006027, ss_p=0.008
Dense epoch 080: train=0.009692, val=0.006034, ss_p=0.010
Dense epoch 08

Training LSTM:   0%|          | 0/500 [00:00<?, ?it/s]

LSTM epoch 001: train=0.009047, val=0.006135, ss_p=0.000
LSTM epoch 005: train=0.007869, val=0.006119, ss_p=0.000
LSTM epoch 010: train=0.007841, val=0.006085, ss_p=0.000
LSTM epoch 015: train=0.007825, val=0.006073, ss_p=0.000
LSTM epoch 020: train=0.007807, val=0.006039, ss_p=0.000
LSTM epoch 025: train=0.007777, val=0.006053, ss_p=0.000
LSTM epoch 030: train=0.007761, val=0.006061, ss_p=0.000
LSTM epoch 035: train=0.007739, val=0.006074, ss_p=0.000
LSTM epoch 040: train=0.007715, val=0.006097, ss_p=0.000
LSTM: early stopping at epoch 41

Transformer runtime shape validation
  X batch shape: (256, 20, 29)
  Y batch shape: (256, 5)


Training Transformer:   0%|          | 0/500 [00:00<?, ?it/s]

Transformer epoch 001: train=0.010780, val=0.006722, ss_p=0.000
Transformer epoch 005: train=0.007862, val=0.006245, ss_p=0.000
Transformer epoch 010: train=0.007860, val=0.006102, ss_p=0.000
Transformer epoch 015: train=0.007828, val=0.006194, ss_p=0.000
Transformer epoch 020: train=0.007822, val=0.006069, ss_p=0.000
Transformer epoch 025: train=0.007844, val=0.006301, ss_p=0.000
Transformer epoch 030: train=0.007838, val=0.006029, ss_p=0.000
Transformer epoch 035: train=0.007894, val=0.006247, ss_p=0.000
Transformer epoch 040: train=0.007875, val=0.006139, ss_p=0.000
Transformer epoch 045: train=0.007879, val=0.006121, ss_p=0.000
Transformer epoch 050: train=0.007892, val=0.006174, ss_p=0.000
Transformer: early stopping at epoch 50

Training complete
  Models trained: ['Dense', 'LSTM', 'Transformer']
  Model failures: {}


In [20]:
# ============================================================================
# Prediction, Ensemble, and Metrics (multi-horizon, quantile-aware)
# ============================================================================

def predict_quantile_model(model, loader, *, device, use_amp, pin_memory):
    """Run model on a loader and return (preds, actuals) where
    preds: (N, Q, H), actuals: (N, H)."""
    model.eval()
    preds_chunks, y_chunks = [], []
    def _ac():
        return torch.cuda.amp.autocast() if use_amp else nullcontext()
    with torch.no_grad():
        for batch_i, (Xb, Yb) in enumerate(loader, start=1):
            Xb = Xb.to(device, non_blocking=pin_memory)
            assert_finite_tensor(f"prediction X batch {batch_i}", Xb)
            with _ac():
                out = model(Xb)
            assert_finite_tensor(f"prediction output batch {batch_i}", out)
            preds_chunks.append(out.detach().cpu().numpy().astype(np.float32))
            y_chunks.append(Yb.detach().cpu().numpy().astype(np.float32))
    if not preds_chunks:
        return (np.empty((0, N_QUANTILES, N_HORIZONS), dtype=np.float32),
                np.empty((0, N_HORIZONS), dtype=np.float32))
    preds = np.concatenate(preds_chunks, axis=0)
    if not np.isfinite(preds).all():
        raise FloatingPointError("Model predictions contain non-finite values")
    return preds, np.concatenate(y_chunks, axis=0)

def normalize_ensemble_weights(model_names, weights=None):
    names = list(model_names)
    if weights is None:
        return {n: 1.0 / len(names) for n in names}
    avail = {n: max(0.0, float(weights.get(n, 0.0))) for n in names}
    total = sum(avail.values())
    if total <= 0:
        return {n: 1.0 / len(names) for n in names}
    return {n: w / total for n, w in avail.items()}

def ensemble_quantile_predict(trained_models, loader, weights, *, device, use_amp, pin_memory):
    """Weighted average of per-model (B,Q,H) outputs. Returns
    (ensemble_preds_BQH, actuals_BH, per_model_preds_dict, used_weights)."""
    used_weights = normalize_ensemble_weights(trained_models.keys(), weights)
    per_model_preds = {}
    actuals = None
    for name, model in trained_models.items():
        preds, y = predict_quantile_model(model, loader,
                                            device=device, use_amp=use_amp,
                                            pin_memory=pin_memory)
        per_model_preds[name] = preds
        if actuals is None:
            actuals = y
        elif actuals.shape != y.shape:
            raise ValueError(f"Actuals shape mismatch for {name}: {y.shape} vs {actuals.shape}")
    ensemble = np.zeros_like(next(iter(per_model_preds.values())))
    for name in trained_models:
        if not np.isfinite(per_model_preds[name]).all():
            raise FloatingPointError(f"{name} predictions contain non-finite values")
        ensemble += used_weights[name] * per_model_preds[name]
    ensemble = sort_quantiles_non_crossing(ensemble)
    if not np.isfinite(ensemble).all():
        raise FloatingPointError("Ensemble predictions contain non-finite values")
    return ensemble, actuals, per_model_preds, used_weights


# ----------------------------------------------------------------------------
# Recursive one-step rollout helpers for live inference
# ----------------------------------------------------------------------------
def _numeric_or_nan(value):
    try:
        if isinstance(value, str):
            value = value.replace(",", "")
        number = float(value)
    except (TypeError, ValueError):
        return np.nan
    return number if np.isfinite(number) else np.nan

def _to_numeric_feature_frame(df):
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == object:
            out[col] = out[col].astype(str).str.replace(",", "", regex=False)
    out = out.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return out.ffill().fillna(0)

def _series_value(row, column):
    if row is None or column not in row.index:
        return np.nan
    return _numeric_or_nan(row[column])

def _last_finite_value(df, column, fallback=np.nan):
    if df.empty or column not in df.columns:
        return fallback
    values = pd.to_numeric(df[column], errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)
    values = values[np.isfinite(values)]
    return float(values[-1]) if len(values) else fallback

def _next_rollout_date(history, date_col):
    if not date_col or history.empty or date_col not in history.columns:
        return pd.NaT
    dates = pd.to_datetime(history[date_col], errors="coerce").dropna()
    if dates.empty:
        return pd.NaT
    return dates.iloc[-1] + pd.offsets.BDay(1)

def _scale_single_feature_value(feature_columns, feature_scaler, column, value):
    if column not in feature_columns:
        return None
    frame = pd.DataFrame(np.zeros((1, len(feature_columns)), dtype=np.float32), columns=list(feature_columns))
    frame.loc[:, column] = float(value)
    scaled = feature_scaler.transform(frame).astype(np.float32)
    return float(scaled[0, list(feature_columns).index(column)])

def _rolling_finite_mean(history, column, window=20, fallback=np.nan):
    if history.empty or column not in history.columns:
        return fallback
    values = pd.to_numeric(history[column], errors="coerce").to_numpy(dtype=np.float64)
    finite = values[np.isfinite(values)]
    if len(finite) == 0:
        return fallback
    return float(np.mean(finite[-window:]))

def _recent_range_fraction(history, fallback=0.02):
    required = {"High", "Low", TARGET_BASE_COL}
    if history.empty or not required.issubset(set(history.columns)):
        return fallback
    high = pd.to_numeric(history["High"], errors="coerce").to_numpy(dtype=np.float64)
    low = pd.to_numeric(history["Low"], errors="coerce").to_numpy(dtype=np.float64)
    close = pd.to_numeric(history[TARGET_BASE_COL], errors="coerce").to_numpy(dtype=np.float64)
    valid = np.isfinite(high) & np.isfinite(low) & np.isfinite(close) & (close > 0) & (high >= low)
    if not valid.any():
        return fallback
    ranges = (high[valid] - low[valid]) / close[valid]
    ranges = ranges[np.isfinite(ranges) & (ranges >= 0)]
    if len(ranges) == 0:
        return fallback
    return float(np.clip(np.median(ranges[-20:]), 0.005, 0.05))

def _stabilized_high_low(history, open_value, close_value):
    reference = max(abs(float(open_value)), abs(float(close_value)), 1.0)
    range_margin = reference * _recent_range_fraction(history) * 0.5
    body_margin = abs(float(open_value) - float(close_value)) * 0.25
    margin = max(range_margin, body_margin)
    upper_base = max(float(open_value), float(close_value))
    lower_base = min(float(open_value), float(close_value))
    high_value = min(upper_base + margin, upper_base * 1.08)
    low_value = max(lower_base - margin, lower_base * 0.92, 0.01)
    if low_value > high_value:
        high_value = upper_base
        low_value = max(lower_base, 0.01)
    return float(high_value), float(low_value)

def _latest_scaled_feature_row(history, feature_columns, feature_scaler, date_col):
    engineered = compute_indicators(history, date_col)
    engineered.columns = [canonical_column_name(col) for col in engineered.columns]
    engineered = engineered.loc[:, ~pd.Index(engineered.columns).duplicated()].copy()
    forbidden = {TARGET_BASE_COL, HAS_LABELS_COL, *LABEL_COLS}
    drop_cols = [c for c in engineered.columns if c in forbidden or is_forward_label_column(c)]
    engineered = engineered.drop(columns=drop_cols, errors="ignore")
    if engineered.empty:
        raise RuntimeError("recursive feature engineering produced no rows")
    feature_frame = engineered.reindex(columns=list(feature_columns), fill_value=0)
    feature_frame = _to_numeric_feature_frame(feature_frame)
    scaled = feature_scaler.transform(feature_frame).astype(np.float32)
    latest = scaled[-1]
    if latest.shape != (len(feature_columns),):
        raise ValueError(f"latest feature row shape mismatch: {latest.shape}")
    if not np.isfinite(latest).all():
        raise FloatingPointError("latest recursive feature row contains non-finite values")
    return latest

def build_latest_sequence_from_history(history_df, feature_columns, feature_scaler, seq_len, date_col=None):
    history = history_df.copy().reset_index(drop=True)
    history.columns = [canonical_column_name(col) for col in history.columns]
    engineered = compute_indicators(history, date_col)
    engineered.columns = [canonical_column_name(col) for col in engineered.columns]
    engineered = engineered.loc[:, ~pd.Index(engineered.columns).duplicated()].copy()
    forbidden = {TARGET_BASE_COL, HAS_LABELS_COL, *LABEL_COLS}
    drop_cols = [c for c in engineered.columns if c in forbidden or is_forward_label_column(c)]
    engineered = engineered.drop(columns=drop_cols, errors="ignore")
    feature_frame = engineered.reindex(columns=list(feature_columns), fill_value=0)
    feature_frame = _to_numeric_feature_frame(feature_frame)
    scaled = feature_scaler.transform(feature_frame).astype(np.float32)
    if len(scaled) < seq_len:
        raise ValueError(f"need at least {seq_len} engineered rows, got {len(scaled)}")
    return scaled[-seq_len:].astype(np.float32)

def predict_quantile_array(trained_models, X, weights, *, device, use_amp):
    X = np.asarray(X, dtype=np.float32)
    if X.ndim != 3:
        raise ValueError(f"X must be 3D, got {X.shape}")
    used_weights = normalize_ensemble_weights(trained_models.keys(), weights)
    per_model_preds = {}
    with torch.no_grad():
        X_tensor = torch.as_tensor(X, dtype=torch.float32, device=device)
        for name, model in trained_models.items():
            model.eval()
            with (torch.cuda.amp.autocast() if use_amp else nullcontext()):
                out = model(X_tensor)
            preds = out.detach().cpu().numpy().astype(np.float32)
            if preds.ndim != 3 or preds.shape[1:] != (N_QUANTILES, N_HORIZONS):
                raise ValueError(f"{name}: expected (*, {N_QUANTILES}, {N_HORIZONS}), got {preds.shape}")
            if not np.isfinite(preds).all():
                raise FloatingPointError(f"{name} recursive predictions contain non-finite values")
            per_model_preds[name] = preds
    ensemble = np.zeros_like(next(iter(per_model_preds.values())))
    for name, preds in per_model_preds.items():
        ensemble += used_weights[name] * preds
    ensemble = sort_quantiles_non_crossing(ensemble)
    return ensemble, per_model_preds, used_weights

def _rollout_observation_row(template_columns, history, observed_row, date_col, open_value, predicted_close):
    row = {col: np.nan for col in template_columns}
    if observed_row is not None:
        row.update(observed_row.to_dict())
    actual_close = _series_value(observed_row, TARGET_BASE_COL)
    close_value = float(actual_close) if np.isfinite(actual_close) else float(predicted_close)
    high_value, low_value = _stabilized_high_low(history, open_value, close_value)
    volume_value = _rolling_finite_mean(history, "Volume", fallback=np.nan)
    if not np.isfinite(volume_value):
        volume_value = _last_finite_value(history, "Volume", 0.0)
    row["Open"] = float(open_value)
    row["High"] = float(high_value)
    row["Low"] = float(low_value)
    row[TARGET_BASE_COL] = close_value
    row["Volume"] = float(volume_value)
    if date_col:
        if observed_row is not None and date_col in observed_row.index and pd.notna(observed_row[date_col]):
            row[date_col] = observed_row[date_col]
        else:
            row[date_col] = _next_rollout_date(history, date_col)
    return pd.DataFrame([row], columns=list(template_columns))

def recursive_close_forecast(
    trained_models,
    last_sequence,
    history_df,
    feature_columns,
    feature_scaler,
    weights,
    *,
    date_col=None,
    future_rows=None,
    steps=N_HORIZONS,
    target_base_col=TARGET_BASE_COL,
    quantiles=QUANTILES,
    device=device,
    use_amp=USE_AMP,
):
    """
    Hybrid q50 close forecast: recursive one-step rollout for the five-day path.

    The first prediction uses last_sequence, which must contain only rows known
    before T+1. After each one-step prediction, the close used for the next
    window is the actual close from future_rows when available; otherwise it is
    the previous prediction. Missing future Open falls back to the previous
    close. The scaler is never refit inside the rollout.
    """
    steps = int(steps)
    if steps < 1 or steps > N_HORIZONS:
        raise ValueError(f"steps must be in 1..{N_HORIZONS}")
    history = history_df.copy().reset_index(drop=True)
    history.columns = [canonical_column_name(col) for col in history.columns]
    if date_col is not None:
        date_col = canonical_column_name(date_col)
    future = pd.DataFrame() if future_rows is None else future_rows.copy().reset_index(drop=True)
    if not future.empty:
        future.columns = [canonical_column_name(col) for col in future.columns]
    feature_columns = list(feature_columns)
    sequence = np.asarray(last_sequence, dtype=np.float32).copy()
    expected_shape = (SEQ_LEN, len(feature_columns))
    if sequence.shape != expected_shape:
        raise ValueError(f"last_sequence must have shape {expected_shape}, got {sequence.shape}")
    q50_idx = quantiles.index(0.50) if 0.50 in quantiles else len(quantiles) // 2
    previous_close = _last_finite_value(history, target_base_col)
    if not np.isfinite(previous_close):
        raise ValueError("history_df must contain a finite previous close")
    template_columns = list(history.columns)
    for col in ["Open", "High", "Low", target_base_col, "Volume"]:
        if col not in template_columns:
            template_columns.append(col)
    if date_col and date_col not in template_columns:
        template_columns.append(date_col)

    # The first forecast step predicts the current row from data through T-1,
    # so Open[T] is unavailable by contract and falls back to previous Close.
    first_open = previous_close
    direct_sequence = sequence.copy()
    scaled_open = _scale_single_feature_value(feature_columns, feature_scaler, "Open", first_open)
    if scaled_open is not None:
        direct_sequence[-1, feature_columns.index("Open")] = scaled_open
    direct_ensemble, _, used_weights = predict_quantile_array(
        trained_models,
        direct_sequence.reshape(1, SEQ_LEN, len(feature_columns)),
        weights,
        device=device,
        use_amp=use_amp,
    )
    direct_logrets = direct_ensemble[0, q50_idx, :steps].astype(np.float64)
    forecast_logrets = [float(value) for value in direct_logrets]
    forecast_prices = [float(previous_close * math.exp(value)) for value in forecast_logrets]

    recursive_steps = min(RECURSIVE_TRAIN_HORIZONS, steps)
    for step in range(recursive_steps):
        observed_row = future.iloc[step] if step < len(future) else None
        if step == 0:
            open_value = previous_close
        else:
            open_value = _series_value(observed_row, "Open")
            if not np.isfinite(open_value):
                open_value = previous_close
        model_sequence = sequence.copy()
        scaled_open = _scale_single_feature_value(feature_columns, feature_scaler, "Open", open_value)
        if scaled_open is not None:
            model_sequence[-1, feature_columns.index("Open")] = scaled_open
        ensemble, _, used_weights = predict_quantile_array(
            trained_models,
            model_sequence.reshape(1, SEQ_LEN, len(feature_columns)),
            weights,
            device=device,
            use_amp=use_amp,
        )
        one_step_logret = float(np.clip(ensemble[0, q50_idx, 0], *SCHEDULED_SAMPLING_RETURN_CLIP))
        predicted_close = float(previous_close * math.exp(one_step_logret))
        if not np.isfinite(predicted_close):
            raise FloatingPointError("recursive forecast produced a non-finite close")
        forecast_prices[step] = predicted_close
        forecast_logrets[step] = one_step_logret

        if step + 1 >= recursive_steps:
            continue
        next_row = _rollout_observation_row(
            template_columns,
            history,
            observed_row,
            date_col,
            open_value,
            predicted_close,
        )
        history = pd.concat([history, next_row], ignore_index=True)
        previous_close = float(next_row.iloc[0][target_base_col])
        latest_scaled = _latest_scaled_feature_row(history, feature_columns, feature_scaler, date_col)
        sequence = np.concatenate([sequence[1:], latest_scaled.reshape(1, -1)], axis=0).astype(np.float32)

    result = {f"Forecast_Close_T+{i + 1}": price for i, price in enumerate(forecast_prices)}
    result.update({f"Forecast_LogRet_q50_T+{i + 1}": value for i, value in enumerate(forecast_logrets)})
    return pd.DataFrame([result]), used_weights

def recursive_forecast_latest_for_stock(symbol, trained_models, per_stock_artifacts, weights, *, steps=N_HORIZONS):
    art = per_stock_artifacts[symbol]
    history = art["source_frame"].reset_index(drop=True).copy()
    date_col = art["date_col"]
    feature_scaler = art["feature_scaler"]
    last_sequence = build_latest_sequence_from_history(
        history,
        common_features,
        feature_scaler,
        SEQ_LEN,
        date_col=date_col,
    )
    forecasts, used_weights = recursive_close_forecast(
        trained_models,
        last_sequence,
        history,
        common_features,
        feature_scaler,
        weights,
        date_col=date_col,
        future_rows=None,
        steps=steps,
        device=device,
        use_amp=USE_AMP,
    )
    forecasts.insert(0, "Symbol", symbol)
    return forecasts, used_weights

def mean_spearman_by_group(y_true, y_pred, groups):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    groups = np.asarray(groups, dtype=object)
    valid = np.isfinite(y_true) & np.isfinite(y_pred) & pd.notna(groups)
    if valid.sum() < 2:
        return None, 0
    tmp = pd.DataFrame({"y": y_true[valid], "p": y_pred[valid], "group": groups[valid]})
    values = []
    for _, sub in tmp.groupby("group"):
        if len(sub) < 2 or sub["y"].nunique() < 2 or sub["p"].nunique() < 2:
            continue
        corr = spearmanr(sub["p"], sub["y"]).correlation
        if np.isfinite(corr):
            values.append(float(corr))
    if not values:
        return None, 0
    return float(np.mean(values)), len(values)

def anchor_dates_for_indices(symbols, target_indices, artifacts):
    dates = []
    for symbol, target_idx in zip(symbols, target_indices):
        art = artifacts[symbol]
        df = art["source_frame"].reset_index(drop=True)
        dates.append(pd.Timestamp(df.iloc[int(target_idx)][art["date_col"]]).normalize())
    return np.asarray(dates, dtype=object)

def compute_horizon_metrics(actuals_BH, preds_BQH, quantiles, horizons, *, label_mask=None, groups=None):
    """Per-horizon: pinball, MAE, coverage, and cross-sectional Spearman IC."""
    q10_i = quantiles.index(0.10) if 0.10 in quantiles else 0
    q50_i = quantiles.index(0.50) if 0.50 in quantiles else len(quantiles) // 2
    q90_i = quantiles.index(0.90) if 0.90 in quantiles else len(quantiles) - 1
    if label_mask is None:
        label_mask = np.ones_like(actuals_BH, dtype=bool)
    rows = []
    for h_i, h in enumerate(horizons):
        y = actuals_BH[:, h_i]
        p = preds_BQH[:, :, h_i]
        valid = label_mask[:, h_i] & np.isfinite(y) & np.isfinite(p).all(axis=1)
        if not valid.any():
            rows.append({
                "horizon": h, "n": 0, "pinball": None, "mae_q50": None,
                "coverage_q10_q90": None, "spearman_ic": None, "spearman_groups": 0,
            })
            continue
        y_v = y[valid]
        p_v = p[valid]
        pin = 0.0
        for q_i, q in enumerate(quantiles):
            diff = y_v - p_v[:, q_i]
            pin += float(np.mean(np.maximum(q * diff, (q - 1) * diff)))
        pin /= len(quantiles)
        mae_q50 = float(np.mean(np.abs(y_v - p_v[:, q50_i])))
        in_band = (y_v >= p_v[:, q10_i]) & (y_v <= p_v[:, q90_i])
        cov = float(in_band.mean())
        if groups is not None:
            ic, ic_groups = mean_spearman_by_group(y_v, p_v[:, q50_i], np.asarray(groups, dtype=object)[valid])
        else:
            corr = spearmanr(p_v[:, q50_i], y_v).correlation if len(y_v) > 1 else np.nan
            ic = float(corr) if np.isfinite(corr) else None
            ic_groups = 1 if ic is not None else 0
        rows.append({
            "horizon": int(h), "n": int(valid.sum()),
            "pinball": pin, "mae_q50": mae_q50,
            "coverage_q10_q90": cov,
            "spearman_ic": ic,
            "spearman_groups": int(ic_groups),
        })
    return rows

def q50_predictions(preds_BQH, quantiles):
    q50_i = quantiles.index(0.50) if 0.50 in quantiles else len(quantiles) // 2
    return preds_BQH[:, q50_i, :]

def compute_decile_lift(actuals_BH, preds_BQH, quantiles, horizons, *, label_mask=None, target_horizons=None):
    if target_horizons is None:
        target_horizons = sorted({horizons[0], horizons[-1]})
    q50 = q50_predictions(preds_BQH, quantiles)
    if label_mask is None:
        label_mask = np.ones_like(actuals_BH, dtype=bool)
    rows = []
    for h in target_horizons:
        if h not in horizons:
            continue
        h_i = horizons.index(h)
        valid = label_mask[:, h_i] & np.isfinite(actuals_BH[:, h_i]) & np.isfinite(q50[:, h_i])
        if valid.sum() < 10:
            rows.append({"horizon": h, "n": int(valid.sum()), "top_decile_mean": None,
                         "bottom_decile_mean": None, "overall_mean": None,
                         "top_vs_overall": None, "top_vs_bottom": None})
            continue
        df_lift = pd.DataFrame({"actual": actuals_BH[valid, h_i], "pred": q50[valid, h_i]})
        df_lift["decile"] = pd.qcut(df_lift["pred"].rank(method="first"), 10, labels=False)
        top = df_lift.loc[df_lift["decile"] == 9, "actual"]
        bottom = df_lift.loc[df_lift["decile"] == 0, "actual"]
        overall = float(df_lift["actual"].mean())
        top_mean = float(top.mean())
        bottom_mean = float(bottom.mean())
        rows.append({
            "horizon": int(h), "n": int(len(df_lift)),
            "top_decile_mean": top_mean,
            "bottom_decile_mean": bottom_mean,
            "overall_mean": overall,
            "top_vs_overall": top_mean - overall,
            "top_vs_bottom": top_mean - bottom_mean,
        })
    return pd.DataFrame(rows)

ensemble_weights = normalized_model_weights(selected_hparams)

test_anchor_dates = anchor_dates_for_indices(test_symbols, test_target_indices, per_stock_artifacts)

# --- Test set (post-train, pre-backtest window): real metrics live here ---
test_ens_BQH, test_actuals_BH, _, used_weights = ensemble_quantile_predict(
    trained_models, test_loader, ensemble_weights,
    device=device, use_amp=USE_AMP, pin_memory=PIN_MEMORY,
)
test_horizon_metrics = compute_horizon_metrics(
    test_actuals_BH, test_ens_BQH, QUANTILES, HORIZONS,
    label_mask=test_label_mask, groups=test_anchor_dates,
)
test_metrics_df = pd.DataFrame(test_horizon_metrics)
test_decile_lift_df = compute_decile_lift(
    test_actuals_BH, test_ens_BQH, QUANTILES, HORIZONS,
    label_mask=test_label_mask,
)
print("\nTest-set per-horizon metrics:")
print(test_metrics_df.head().to_string(index=False))
print("...")
print(test_metrics_df.tail().to_string(index=False))
print(f"\nMean test pinball : {test_metrics_df['pinball'].mean():.6f}")
print(f"Mean coverage     : {test_metrics_df['coverage_q10_q90'].mean():.3f} (target ~0.80)")
print(f"Mean Spearman IC  : {test_metrics_df['spearman_ic'].mean():.4f}")
print(f"Ensemble weights used: {used_weights}")
print("\nTest decile lift:")
print(test_decile_lift_df.to_string(index=False))



Test-set per-horizon metrics:
 horizon     n  pinball  mae_q50  coverage_q10_q90  spearman_ic  spearman_groups
       1 18081 0.004005 0.011759          0.874288     0.009217              369
       2 18032 0.005648 0.016884          0.864186     0.002689              368
       3 17983 0.006969 0.021073          0.860146     0.009531              367
       4 17934 0.008057 0.024530          0.849894     0.007197              366
       5 17885 0.009017 0.027560          0.847973     0.046396              365
...
 horizon     n  pinball  mae_q50  coverage_q10_q90  spearman_ic  spearman_groups
       1 18081 0.004005 0.011759          0.874288     0.009217              369
       2 18032 0.005648 0.016884          0.864186     0.002689              368
       3 17983 0.006969 0.021073          0.860146     0.009531              367
       4 17934 0.008057 0.024530          0.849894     0.007197              366
       5 17885 0.009017 0.027560          0.847973     0.046396           

In [21]:
# ============================================================================
# Sequential Backtest (BACK_TEST_START .. BACK_TEST_END)
# ============================================================================
# This is a strict walk-forward evaluation on the backtest window.
# Same no-leakage contract as training/test:
#   - Prediction for anchor day t uses ONLY rows [t-SEQ_LEN, t-1].
#   - The model never sees row t (no Open/High/Low/Close/Volume/indicators of t).
#   - Per-stock feature scaler is the one fit on TRAIN (frozen).
# Output: a long-format DataFrame with one row per (stock, anchor_date, horizon).
# ============================================================================

def run_backtest(
    trained_models, per_stock_artifacts, weights, *,
    feature_columns, label_cols, seq_len, horizons, quantiles,
    target_base_col, device, use_amp, pin_memory, batch_size,
):
    used_weights = normalize_ensemble_weights(trained_models.keys(), weights)
    all_rows = []

    for symbol, art in tqdm(per_stock_artifacts.items(), desc="Backtest"):
        idx = np.asarray(art["backtest_target_indices"], dtype=np.int64)
        if len(idx) == 0:
            continue

        df = art["source_frame"].reset_index(drop=True)
        date_col = art["date_col"]
        fscaler = art["feature_scaler"]
        label_mask = horizon_realisation_mask(df, idx, date_col, horizons, BACK_TEST_END)

        # Build sequences for backtest anchor rows using the frozen TRAIN scaler.
        X_bk, _, valid_idx, valid_label_mask, _ = build_sequences_for_target_indices(
            df, feature_columns, label_cols, idx, seq_len,
            fscaler, label_valid_mask=label_mask, require_full_labels=False,
        )
        if len(X_bk) == 0:
            continue

        ds = StockMultiHorizonDataset(
            X_bk, np.zeros((len(X_bk), len(label_cols)), dtype=np.float32),
            seq_len, len(feature_columns), len(label_cols), require_finite_y=True,
        )
        loader = make_loader(ds, batch_size, NUM_WORKERS, pin_memory)

        ens, _, _, _ = ensemble_quantile_predict(
            trained_models, loader, weights,
            device=device, use_amp=use_amp, pin_memory=pin_memory,
        )  # ens: (N, Q, H)

        # Convert q50 log-return path to predicted price path. Labels are
        # y_h = log(Close[t+h-1] / Close[t-1]), so the anchor is Close[t-1].
        # IMPORTANT: Close[t] is NEVER fed to the model input.
        anchor_positions = np.asarray(valid_idx, dtype=np.int64) - 1
        if (anchor_positions < 0).any():
            raise RuntimeError("Backtest anchor positions must be at least 1")
        anchor_close = df.iloc[anchor_positions][target_base_col].to_numpy(dtype=np.float64)
        dates = pd.to_datetime(df.iloc[valid_idx][date_col]).reset_index(drop=True)

        q10_i = quantiles.index(0.10)
        q50_i = quantiles.index(0.50)
        q90_i = quantiles.index(0.90)

        realised = df.iloc[valid_idx][label_cols].to_numpy(dtype=np.float64)

        for i, t in enumerate(valid_idx):
            for h_i, h in enumerate(horizons):
                q10 = float(ens[i, q10_i, h_i])
                q50 = float(ens[i, q50_i, h_i])
                q90 = float(ens[i, q90_i, h_i])
                ac = float(anchor_close[i])
                price_q50 = ac * math.exp(q50)
                price_q10 = ac * math.exp(q10)
                price_q90 = ac * math.exp(q90)
                realised_h = realised[i, h_i]
                realised_ok = bool(valid_label_mask[i, h_i]) and np.isfinite(realised_h)
                all_rows.append({
                    "Symbol": symbol,
                    "Anchor_Index": int(t),
                    "Anchor_Date": dates.iloc[i],
                    "Horizon": int(h),
                    "Anchor_Close": ac,
                    "Pred_LogRet_q10": q10,
                    "Pred_LogRet_q50": q50,
                    "Pred_LogRet_q90": q90,
                    "Pred_Price_q10": price_q10,
                    "Pred_Price_q50": price_q50,
                    "Pred_Price_q90": price_q90,
                    "Realised_In_Split": realised_ok,
                    "Realised_LogRet": float(realised_h) if realised_ok else None,
                    "Realised_Price": (ac * math.exp(realised_h)) if realised_ok else None,
                })
    return pd.DataFrame(all_rows), used_weights

backtest_results_df, backtest_weights = run_backtest(
    trained_models, per_stock_artifacts, ensemble_weights,
    feature_columns=common_features, label_cols=LABEL_COLS,
    seq_len=SEQ_LEN, horizons=HORIZONS, quantiles=QUANTILES,
    target_base_col=TARGET_BASE_COL,
    device=device, use_amp=USE_AMP, pin_memory=PIN_MEMORY,
    batch_size=BATCH_SIZE,
)

if backtest_results_df.empty:
    print("Backtest produced zero rows — check BACK_TEST_START/END.")
    backtest_metrics_df = pd.DataFrame()
    backtest_decile_lift_df = pd.DataFrame()
else:
    print(f"Backtest rows: {len(backtest_results_df)}")
    print(backtest_results_df.head().to_string(index=False))

    bt_eval = backtest_results_df.dropna(subset=["Realised_LogRet"]).copy()
    if not bt_eval.empty:
        bt_metrics = []
        for h in HORIZONS:
            sub = bt_eval[bt_eval["Horizon"] == h]
            if sub.empty:
                continue
            y = sub["Realised_LogRet"].to_numpy(dtype=np.float64)
            q10 = sub["Pred_LogRet_q10"].to_numpy(dtype=np.float64)
            q50 = sub["Pred_LogRet_q50"].to_numpy(dtype=np.float64)
            q90 = sub["Pred_LogRet_q90"].to_numpy(dtype=np.float64)
            valid = np.isfinite(y) & np.isfinite(q10) & np.isfinite(q50) & np.isfinite(q90)
            y, q10, q50, q90 = y[valid], q10[valid], q50[valid], q90[valid]
            if len(y) == 0:
                continue
            pin = 0.0
            for q, pred in [(0.10, q10), (0.50, q50), (0.90, q90)]:
                diff = y - pred
                pin += float(np.mean(np.maximum(q * diff, (q - 1) * diff)))
            pin /= 3.0
            cov = float(((y >= q10) & (y <= q90)).mean())
            mae = float(np.mean(np.abs(y - q50)))
            ic, ic_groups = mean_spearman_by_group(
                y, q50, sub.loc[valid, "Anchor_Date"].to_numpy(dtype=object)
            )
            bt_metrics.append({
                "horizon": int(h), "n": int(len(y)),
                "pinball": pin, "mae_q50": mae,
                "coverage_q10_q90": cov,
                "spearman_ic": ic,
                "spearman_groups": int(ic_groups),
            })
        backtest_metrics_df = pd.DataFrame(bt_metrics)
        print("\nBacktest per-horizon metrics (head/tail):")
        print(backtest_metrics_df.head().to_string(index=False))
        print(backtest_metrics_df.tail().to_string(index=False))

        bt_actual = np.full((len(backtest_results_df["Anchor_Date"].drop_duplicates()), N_HORIZONS), np.nan)
        backtest_decile_rows = []
        for h in sorted({HORIZONS[0], HORIZONS[-1]}):
            sub = bt_eval[bt_eval["Horizon"] == h].copy()
            sub = sub[np.isfinite(sub["Realised_LogRet"]) & np.isfinite(sub["Pred_LogRet_q50"])]
            if len(sub) < 10:
                backtest_decile_rows.append({"horizon": h, "n": int(len(sub)), "top_decile_mean": None,
                                             "bottom_decile_mean": None, "overall_mean": None,
                                             "top_vs_overall": None, "top_vs_bottom": None})
                continue
            sub["decile"] = pd.qcut(sub["Pred_LogRet_q50"].rank(method="first"), 10, labels=False)
            top = sub.loc[sub["decile"] == 9, "Realised_LogRet"]
            bottom = sub.loc[sub["decile"] == 0, "Realised_LogRet"]
            overall = float(sub["Realised_LogRet"].mean())
            top_mean = float(top.mean())
            bottom_mean = float(bottom.mean())
            backtest_decile_rows.append({
                "horizon": int(h), "n": int(len(sub)),
                "top_decile_mean": top_mean,
                "bottom_decile_mean": bottom_mean,
                "overall_mean": overall,
                "top_vs_overall": top_mean - overall,
                "top_vs_bottom": top_mean - bottom_mean,
            })
        backtest_decile_lift_df = pd.DataFrame(backtest_decile_rows)
        print("\nBacktest decile lift:")
        print(backtest_decile_lift_df.to_string(index=False))
    else:
        print("No realised labels available in backtest window — metrics skipped.")
        backtest_metrics_df = pd.DataFrame()
        backtest_decile_lift_df = pd.DataFrame()


Backtest:   0%|          | 0/49 [00:00<?, ?it/s]

Backtest rows: 30870
  Symbol  Anchor_Index Anchor_Date  Horizon  Anchor_Close  Pred_LogRet_q10  Pred_LogRet_q50  Pred_LogRet_q90  Pred_Price_q10  Pred_Price_q50  Pred_Price_q90  Realised_In_Split  Realised_LogRet  Realised_Price
RELIANCE          2590  2025-07-01        1   1494.630615        -0.022532         0.000401         0.023164     1461.330778     1495.229464     1529.655845               True         0.018356     1522.320068
RELIANCE          2590  2025-07-01        2   1494.630615        -0.032390        -0.000184         0.030770     1446.995282     1494.356254     1541.334901               True         0.012056     1512.758301
RELIANCE          2590  2025-07-01        3   1494.630615        -0.039663        -0.001723         0.038084     1436.509501     1492.057753     1552.650041               True         0.011397     1511.762207
RELIANCE          2590  2025-07-01        4   1494.630615        -0.044906        -0.001761         0.043656     1428.997374     1492.001052   

In [22]:
# ============================================================================
# Evaluation Metrics (same metric set as model_2(GPU).ipynb)
# ============================================================================

def metric_value(value):
    if value is None or not np.isfinite(value):
        return None
    return float(value)

def calculate_metrics(y_true, y_pred, label, predictor_count=actual_input_size):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    n = len(y_true)
    if n == 0:
        print(f"{label}: no valid rows")
        return {"n": 0, "MSE": None, "RMSE": None, "MAE": None, "R2": None, "Adjusted_R2": None, "MAPE": None}

    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred) if n > 1 and len(np.unique(y_true)) > 1 else np.nan
    adjusted_r2 = np.nan
    if n > predictor_count + 1 and np.isfinite(r2):
        adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - predictor_count - 1)
    nonzero = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100 if nonzero.any() else np.nan

    print(f"{label}: RMSE={rmse:.6f}, MAE={mae:.6f}, R2={r2:.6f}")
    return {
        "n": int(n),
        "MSE": metric_value(mse),
        "RMSE": metric_value(rmse),
        "MAE": metric_value(mae),
        "R2": metric_value(r2),
        "Adjusted_R2": metric_value(adjusted_r2),
        "MAPE": metric_value(mape),
    }

def price_paths_from_log_returns(actuals_BH, preds_q50_BH, symbols, target_indices, artifacts, target_base_col):
    anchor_close = np.empty(len(symbols), dtype=np.float64)
    for i, (symbol, target_idx) in enumerate(zip(symbols, target_indices)):
        df = artifacts[symbol]["source_frame"].reset_index(drop=True)
        anchor_pos = int(target_idx) - 1
        if anchor_pos < 0:
            raise RuntimeError("Price path anchor position must be at least 1")
        anchor_close[i] = float(df.iloc[anchor_pos][target_base_col])
    actual_prices = anchor_close[:, None] * np.exp(actuals_BH)
    predicted_prices = anchor_close[:, None] * np.exp(preds_q50_BH)
    return actual_prices, predicted_prices

def masked_metric_inputs(actuals_BH, preds_BH, label_mask):
    valid = np.asarray(label_mask, dtype=bool) & np.isfinite(actuals_BH) & np.isfinite(preds_BH)
    return actuals_BH[valid], preds_BH[valid]

train_eval_BQH, train_eval_actuals_BH, _, _ = ensemble_quantile_predict(
    trained_models, train_loader, ensemble_weights,
    device=device, use_amp=USE_AMP, pin_memory=PIN_MEMORY,
)
validation_eval_BQH, validation_eval_actuals_BH, _, _ = ensemble_quantile_predict(
    trained_models, inval_loader, ensemble_weights,
    device=device, use_amp=USE_AMP, pin_memory=PIN_MEMORY,
)

train_pred_q50_BH = q50_predictions(train_eval_BQH, QUANTILES)
validation_pred_q50_BH = q50_predictions(validation_eval_BQH, QUANTILES)
test_pred_q50_BH = q50_predictions(test_ens_BQH, QUANTILES)

train_actual_m, train_pred_m = masked_metric_inputs(train_eval_actuals_BH, train_pred_q50_BH, train_label_mask)
val_actual_m, val_pred_m = masked_metric_inputs(validation_eval_actuals_BH, validation_pred_q50_BH, inval_label_mask)
test_actual_m, test_pred_m = masked_metric_inputs(test_actuals_BH, test_pred_q50_BH, test_label_mask)

metrics = {
    "Train": calculate_metrics(train_actual_m, train_pred_m, "Train log-return q50 metrics"),
    "Validation": calculate_metrics(val_actual_m, val_pred_m, "Validation log-return q50 metrics"),
    "Test": calculate_metrics(test_actual_m, test_pred_m, "Test log-return q50 metrics"),
}

test_actual_price_BH, test_pred_price_BH = price_paths_from_log_returns(
    test_actuals_BH, test_pred_q50_BH, test_symbols, test_target_indices,
    per_stock_artifacts, TARGET_BASE_COL,
)
price_actual_m, price_pred_m = masked_metric_inputs(test_actual_price_BH, test_pred_price_BH, test_label_mask)
price_metrics = calculate_metrics(price_actual_m, price_pred_m, "Test price q50 metrics")

metrics_df = pd.DataFrame(metrics).T
price_metrics_df = pd.DataFrame([price_metrics], index=["Test_Price"])

print("\nEvaluation metrics:")
print(metrics_df.to_string())
print("\nPrice metrics:")
print(price_metrics_df.to_string())


Train log-return q50 metrics: RMSE=0.034453, MAE=0.023544, R2=0.004682
Validation log-return q50 metrics: RMSE=0.027232, MAE=0.017413, R2=-0.041139
Test log-return q50 metrics: RMSE=0.028679, MAE=0.020340, R2=-0.009012
Test price q50 metrics: RMSE=137.560307, MAE=54.230557, R2=0.998998

Evaluation metrics:
                   n       MSE      RMSE       MAE        R2  Adjusted_R2        MAPE
Train       463750.0  0.001187  0.034453  0.023544  0.004682     0.003435  111.891192
Validation   53695.0  0.000742  0.027232  0.017413 -0.041139    -0.052509  121.070969
Test         89915.0  0.000822  0.028679  0.020340 -0.009012    -0.015563  114.026130

Price metrics:
                n           MSE        RMSE        MAE        R2  Adjusted_R2      MAPE
Test_Price  89915  18922.838116  137.560307  54.230557  0.998998     0.998991  2.030942


In [23]:
# ============================================================================
# Save Models, Metadata, and Results
# ============================================================================
model_dir = PROJECT_ROOT / "outputs" / "Saved_Models"
model_dir.mkdir(parents=True, exist_ok=True)

model_save_paths = {
    "Dense": model_dir / "Dense.pt",
    "LSTM": model_dir / "LSTM.pt",
    "Transformer": model_dir / "Transformer.pt",
}
missing_models = [name for name in model_save_paths if name not in trained_models]
if missing_models:
    raise RuntimeError(f"Cannot save missing trained models: {missing_models}")

def assert_state_dict_finite(model_name, model):
    for param_name, tensor in model.state_dict().items():
        if not torch.isfinite(tensor).all():
            raise FloatingPointError(f"{model_name}.{param_name} contains non-finite values")

for model_name, path in model_save_paths.items():
    assert_state_dict_finite(model_name, trained_models[model_name])
    torch.save(trained_models[model_name].state_dict(), path)
    print(f"Saved {model_name} (overwritten if existed): {path}")

metadata = {
    "data_path": str(DATA_PATH),
    "seq_len": int(actual_seq_len),
    "feature_count": int(actual_feature_count),
    "dense_input_size": int(actual_input_size),
    "feature_columns": common_features,
    "label_columns": LABEL_COLS,
    "horizons": HORIZONS,
    "quantiles": QUANTILES,
    "target_type": "multi_horizon_quantile_log_return",
    "target_base_col": TARGET_BASE_COL,
    "label_anchor": "previous_close",
    "forecast_horizon_count": int(N_HORIZONS),
    "forecast_column_names": [f"Forecast_Close_T+{h}" for h in HORIZONS],
    "forecasting_strategy": {
        "recursive_horizons": list(range(1, RECURSIVE_TRAIN_HORIZONS + 1)),
        "direct_horizons": list(range(RECURSIVE_TRAIN_HORIZONS + 1, max(HORIZONS) + 1)),
        "direct_recursive_blend": float(selected_hparams.get("direct_recursive_blend", 0.0)),
        "recursive_return_clip": list(SCHEDULED_SAMPLING_RETURN_CLIP),
    },
    "forecast_stabilization": {
        "rolling_window_length": int(ROLLING_STABILIZATION_WINDOW),
        "use_volatility_normalization": bool(USE_VOLATILITY_NORMALIZATION),
        "feature_stabilization": "recursive close update; previous-close open fallback; smoothed volume; bounded high/low",
    },
    "scheduled_sampling": SCHEDULED_SAMPLING_CONFIG,
    "model_capacity": model_capacity,
    "optuna": OPTUNA_RESULTS,
    "splits": {
        "train_start": str(TRAIN_START.date()),
        "train_end":   str(TRAIN_END.date()),
        "test_start":  str(TEST_START.date()),
        "test_end":    str(TEST_END.date()),
        "back_test_start": str(BACK_TEST_START.date()),
        "back_test_end":   str(BACK_TEST_END.date()),
        "embargo_trading_rows": int(EMBARGO_DAYS),
    },
    "sample_counts": {
        "train_fit": int(len(X_train)),
        "validation": int(len(X_inval)),
        "test": int(len(X_test)),
        "backtest": int(len(X_back)),
        "test_metric_horizons": int(test_label_mask.sum()),
        "backtest_metric_horizons": int(back_label_mask.sum()),
    },
    "valid_stocks": sorted(stock_frames.keys()),
    "skipped_stocks": skipped_stocks,
    "sequence_failures": sequence_failures,
    "sequence_filter_stats": sequence_filter_stats,
    "model_failures": model_failures,
    "ensemble_weights": used_weights,
    "metrics_scaled": metrics,
    "test_price_metrics": price_metrics,
    "test_horizon_metrics": test_horizon_metrics,
    "test_decile_lift": test_decile_lift_df.to_dict(orient="records"),
    "backtest_decile_lift": backtest_decile_lift_df.to_dict(orient="records") if not backtest_decile_lift_df.empty else [],
}

metadata_path = OUTPUT_DIR / "pipeline_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"Saved metadata: {metadata_path}")

# Per-horizon test metrics CSV
test_metrics_path = OUTPUT_DIR / "test_horizon_metrics.csv"
test_metrics_df.to_csv(test_metrics_path, index=False)
print(f"Saved test horizon metrics: {test_metrics_path}")

# Backtest predictions (long-format)
if not backtest_results_df.empty:
    bt_path = OUTPUT_DIR / "backtest_predictions.csv"
    backtest_results_df.to_csv(bt_path, index=False)
    print(f"Saved backtest predictions: {bt_path}")
    if not backtest_metrics_df.empty:
        bt_metrics_path = OUTPUT_DIR / "backtest_horizon_metrics.csv"
        backtest_metrics_df.to_csv(bt_metrics_path, index=False)
        print(f"Saved backtest horizon metrics: {bt_metrics_path}")

excel_path = OUTPUT_DIR / "stock_results.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    test_metrics_df.to_excel(writer, sheet_name="Test_Horizon_Metrics", index=False)
    metrics_df.to_excel(writer, sheet_name="Scaled_Metrics")
    price_metrics_df.to_excel(writer, sheet_name="Price_Metrics")
    test_decile_lift_df.to_excel(writer, sheet_name="Test_Decile_Lift", index=False)
    if not backtest_decile_lift_df.empty:
        backtest_decile_lift_df.to_excel(writer, sheet_name="Backtest_Decile_Lift", index=False)
    if not backtest_results_df.empty:
        backtest_results_df.head(200_000).to_excel(
            writer, sheet_name="Backtest_Predictions", index=False
        )
    if not backtest_results_df.empty and not backtest_metrics_df.empty:
        backtest_metrics_df.to_excel(
            writer, sheet_name="Backtest_Horizon_Metrics", index=False
        )
    pd.DataFrame({"Feature": common_features}).to_excel(
        writer, sheet_name="Features", index=False
    )
    pd.DataFrame({"Label": LABEL_COLS}).to_excel(
        writer, sheet_name="Labels", index=False
    )
    pd.DataFrame([{"Stock": k, "Error": v} for k, v in skipped_stocks.items()]).to_excel(
        writer, sheet_name="Skipped_Stocks", index=False
    )
    pd.DataFrame([{"Stock": k, "Error": v} for k, v in sequence_failures.items()]).to_excel(
        writer, sheet_name="Sequence_Failures", index=False
    )
print(f"Saved results workbook: {excel_path}")


Saved Dense (overwritten if existed): /Users/hrishavmajumder/Documents/Stock_Market/outputs/Saved_Models/Dense.pt
Saved LSTM (overwritten if existed): /Users/hrishavmajumder/Documents/Stock_Market/outputs/Saved_Models/LSTM.pt
Saved Transformer (overwritten if existed): /Users/hrishavmajumder/Documents/Stock_Market/outputs/Saved_Models/Transformer.pt
Saved metadata: /Users/hrishavmajumder/Documents/Stock_Market/outputs/pipeline_metadata.json
Saved test horizon metrics: /Users/hrishavmajumder/Documents/Stock_Market/outputs/test_horizon_metrics.csv
Saved backtest predictions: /Users/hrishavmajumder/Documents/Stock_Market/outputs/backtest_predictions.csv
Saved backtest horizon metrics: /Users/hrishavmajumder/Documents/Stock_Market/outputs/backtest_horizon_metrics.csv
Saved results workbook: /Users/hrishavmajumder/Documents/Stock_Market/outputs/stock_results.xlsx


In [ ]:
# # ============================================================================
# # Plotly Interactive Visualization
# # ============================================================================
# plots_dir = OUTPUT_DIR / "plots"
# plots_dir.mkdir(parents=True, exist_ok=True)
# plotly_template = "plotly_white"

# # 1. Training/validation loss curves (pinball loss)
# loss_fig = go.Figure()
# for model_name, history in training_history.items():
#     loss_fig.add_trace(go.Scatter(
#         x=list(range(1, len(history["train"]) + 1)),
#         y=history["train"], mode="lines", name=f"{model_name} Train",
#     ))
#     loss_fig.add_trace(go.Scatter(
#         x=list(range(1, len(history["val"]) + 1)),
#         y=history["val"], mode="lines", name=f"{model_name} In-train Val",
#         line={"dash": "dash"},
#     ))
# loss_fig.update_layout(
#     title="Training and In-train Validation Pinball Loss",
#     xaxis_title="Epoch", yaxis_title="Pinball Loss",
#     template=plotly_template, hovermode="x unified",
# )
# loss_path = plots_dir / "training_validation_loss.html"
# loss_fig.write_html(loss_path); loss_fig.show()

# # 2. Test-set per-horizon coverage and MAE
# if not test_metrics_df.empty:
#     horizon_fig = make_subplots(rows=1, cols=2,
#         subplot_titles=("MAE of q50 vs horizon", "q10-q90 coverage vs horizon"))
#     horizon_fig.add_trace(go.Scatter(
#         x=test_metrics_df["horizon"], y=test_metrics_df["mae_q50"],
#         mode="lines+markers", name="MAE q50"), row=1, col=1)
#     horizon_fig.add_trace(go.Scatter(
#         x=test_metrics_df["horizon"], y=test_metrics_df["coverage_q10_q90"],
#         mode="lines+markers", name="Coverage"), row=1, col=2)
#     horizon_fig.add_hline(y=0.80, line_dash="dash", row=1, col=2,
#                            annotation_text="target 0.80")
#     horizon_fig.update_layout(title="Test-set per-horizon performance",
#                                 template=plotly_template, showlegend=False)
#     hp = plots_dir / "test_horizon_performance.html"
#     horizon_fig.write_html(hp); horizon_fig.show()

# # 3. Sample backtest fan-chart for a single stock (q10/q50/q90 path)
# if not backtest_results_df.empty:
#     sample_symbol = backtest_results_df["Symbol"].iloc[0]
#     anchor_dates = backtest_results_df.loc[
#         backtest_results_df["Symbol"] == sample_symbol, "Anchor_Date"
#     ].drop_duplicates().sort_values()
#     if len(anchor_dates) > 0:
#         sample_anchor = anchor_dates.iloc[len(anchor_dates) // 2]
#         sub = backtest_results_df[
#             (backtest_results_df["Symbol"] == sample_symbol) &
#             (backtest_results_df["Anchor_Date"] == sample_anchor)
#         ].sort_values("Horizon")
#         fan = go.Figure()
#         fan.add_trace(go.Scatter(x=sub["Horizon"], y=sub["Pred_Price_q90"],
#             name="q90", line={"color": "lightblue"}))
#         fan.add_trace(go.Scatter(x=sub["Horizon"], y=sub["Pred_Price_q10"],
#             name="q10", line={"color": "lightblue"}, fill="tonexty"))
#         fan.add_trace(go.Scatter(x=sub["Horizon"], y=sub["Pred_Price_q50"],
#             name="q50", line={"color": "blue"}))
#         if sub["Realised_Price"].notna().any():
#             fan.add_trace(go.Scatter(x=sub["Horizon"], y=sub["Realised_Price"],
#                 name="Realised", line={"color": "black", "dash": "dot"}))
#         fan.update_layout(
#             title=f"Backtest fan chart: {sample_symbol} anchored {sample_anchor}",
#             xaxis_title="Horizon (days ahead)", yaxis_title="Price",
#             template=plotly_template,
#         )
#         fan_path = plots_dir / "backtest_sample_fan_chart.html"
#         fan.write_html(fan_path); fan.show()

# print("Saved Plotly visuals to:", plots_dir)


NameError: name 'training_history' is not defined

In [25]:
# ============================================================================
# Actual vs Predicted Close Price Plot (2025-05-01 to 2026-05-01)
# ============================================================================
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import clear_output, display

START_DATE = pd.Timestamp("2025-05-01")
END_DATE = pd.Timestamp("2026-05-01")
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", Path.cwd()))
OUTPUT_DIR = Path(globals().get("OUTPUT_DIR", PROJECT_ROOT / "outputs"))

PRICE_PREFIXES_FOR_PLOT = {
    "open_": "Open", "high_": "High", "low_": "Low",
    "close_": "Close", "volume_": "Volume",
}
EXACT_PRICE_COLUMNS_FOR_PLOT = {
    "open": "Open", "high": "High", "low": "Low",
    "close": "Close", "volume": "Volume",
}


def _canonical_plot_column(column):
    if "canonical_column_name" in globals():
        return canonical_column_name(column)
    name = str(column).strip()
    lower = name.lower()
    if lower in EXACT_PRICE_COLUMNS_FOR_PLOT:
        return EXACT_PRICE_COLUMNS_FOR_PLOT[lower]
    for prefix, canonical in PRICE_PREFIXES_FOR_PLOT.items():
        if lower.startswith(prefix):
            return canonical
    return name


def _detect_plot_date_column(df):
    if "detect_date_column" in globals():
        detected = detect_date_column(df)
        if detected is not None:
            return detected
    candidates = [c for c in df.columns if "date" in str(c).lower() or "time" in str(c).lower()]
    return candidates[0] if candidates else None


def _load_prediction_data():
    if "backtest_results_df" in globals() and isinstance(backtest_results_df, pd.DataFrame) and not backtest_results_df.empty:
        raw = backtest_results_df.copy()
        source = "backtest_results_df"
    else:
        candidates = [
            OUTPUT_DIR / "backtest_predictions.csv",
            PROJECT_ROOT / "outputs" / "backtest_predictions.csv",
            OUTPUT_DIR / "walk_forward_predictions.csv",
            PROJECT_ROOT / "outputs" / "walk_forward_predictions.csv",
            OUTPUT_DIR / "main_inference" / "predictions.csv",
            PROJECT_ROOT / "outputs" / "main_inference" / "predictions.csv",
        ]
        existing = next((path for path in candidates if path.exists()), None)
        if existing is None:
            raise FileNotFoundError("No prediction output CSV found. Run inference/backtest first.")
        raw = pd.read_csv(existing)
        source = str(existing)

    if {"Symbol", "Anchor_Date", "Horizon", "Pred_Price_q50"}.issubset(raw.columns):
        pred = raw.loc[pd.to_numeric(raw["Horizon"], errors="coerce") == 1].copy()
        pred = pd.DataFrame({
            "Symbol": pred["Symbol"].astype(str),
            "Date": pd.to_datetime(pred["Anchor_Date"], errors="coerce"),
            "Predicted_Close": pd.to_numeric(pred["Pred_Price_q50"], errors="coerce"),
        })
    elif {"Symbol", "Date", "Predicted_Price"}.issubset(raw.columns):
        pred = pd.DataFrame({
            "Symbol": raw["Symbol"].astype(str),
            "Date": pd.to_datetime(raw["Date"], errors="coerce"),
            "Predicted_Close": pd.to_numeric(raw["Predicted_Price"], errors="coerce"),
        })
    else:
        raise ValueError(f"Unsupported prediction schema in {source}")

    pred = pred.dropna(subset=["Symbol", "Date", "Predicted_Close"])
    pred = pred[(pred["Date"] >= START_DATE) & (pred["Date"] <= END_DATE)]
    return pred.sort_values(["Symbol", "Date"]), source


def _find_source_workbook():
    if "DATA_PATH" in globals() and Path(DATA_PATH).exists():
        return Path(DATA_PATH)
    candidates = [
        PROJECT_ROOT / "Data" / "nse_stock_data.xlsx",
        PROJECT_ROOT / "Data" / "DATA_Stock.xlsx",
        PROJECT_ROOT / "nse_stock_data.xlsx",
        PROJECT_ROOT / "DATA_Stock.xlsx",
    ]
    return next((path for path in candidates if path.exists()), None)


def _load_actual_close(symbol):
    if "stock_frames" in globals() and symbol in stock_frames:
        raw = stock_frames[symbol].copy()
    else:
        workbook = _find_source_workbook()
        if workbook is None:
            raise FileNotFoundError("No source workbook found for actual close values.")
        raw = pd.read_excel(workbook, sheet_name=symbol)

    raw = raw.copy()
    raw.columns = [_canonical_plot_column(col) for col in raw.columns]
    raw = raw.loc[:, ~pd.Index(raw.columns).duplicated()].copy()
    date_col = _detect_plot_date_column(raw)
    if date_col is None:
        raise ValueError(f"{symbol}: no date column found")
    if "Close" not in raw.columns:
        raise ValueError(f"{symbol}: no Close column found")

    actual = pd.DataFrame({
        "Date": pd.to_datetime(raw[date_col], errors="coerce"),
        "Actual_Close": pd.to_numeric(raw["Close"], errors="coerce"),
    })
    actual = actual.dropna(subset=["Date", "Actual_Close"])
    actual = actual[(actual["Date"] >= START_DATE) & (actual["Date"] <= END_DATE)]
    return actual.sort_values("Date")


prediction_data, prediction_source = _load_prediction_data()
stock_options = sorted(prediction_data["Symbol"].dropna().astype(str).unique())
if not stock_options:
    raise RuntimeError(f"No predicted close rows found between {START_DATE.date()} and {END_DATE.date()}.")


def plot_stock_close(symbol):
    symbol = str(symbol)
    actual = _load_actual_close(symbol)
    predicted = prediction_data[prediction_data["Symbol"] == symbol].copy()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=actual["Date"],
        y=actual["Actual_Close"],
        mode="lines",
        name="Actual Close",
        line={"color": "#1f77b4", "width": 2},
    ))
    fig.add_trace(go.Scatter(
        x=predicted["Date"],
        y=predicted["Predicted_Close"],
        mode="lines",
        name="Predicted Close",
        line={"color": "#d62728", "width": 2},
    ))
    fig.update_layout(
        title=f"{symbol}: Actual Close vs Predicted Close",
        xaxis_title="Date",
        yaxis_title="Close Price",
        template="plotly_white",
        hovermode="x unified",
        width=1100,
        height=560,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "right", "x": 1},
    )
    fig.update_xaxes(range=[START_DATE, END_DATE])
    fig.show()

    if predicted.empty:
        print(f"No predicted close rows for {symbol} in the selected timeline. Source: {prediction_source}")


try:
    import ipywidgets as widgets

    stock_dropdown = widgets.Dropdown(
        options=stock_options,
        value=stock_options[0],
        description="Stock:",
        layout=widgets.Layout(width="320px"),
    )
    graph_output = widgets.Output()

    def _render_selected_stock(change=None):
        with graph_output:
            clear_output(wait=True)
            plot_stock_close(stock_dropdown.value)

    stock_dropdown.observe(_render_selected_stock, names="value")
    display(stock_dropdown, graph_output)
    _render_selected_stock()
except Exception as exc:
    print(f"Interactive dropdown unavailable ({exc}). Call plot_stock_close('RELIANCE') manually.")
    plot_stock_close(stock_options[0])


Dropdown(description='Stock:', layout=Layout(width='320px'), options=('ADANIENT', 'ADANIPORTS', 'APOLLOHOSP', …

Output()

In [26]:
# ============================================================================
# Final Implementation Summary
# ============================================================================
summary_payload = {
    "sections_modified": [
        "Imports/configuration",
        "Model architecture capacity parameters",
        "Training utility pruning/weight decay support",
        "Optuna walk-forward hyperparameter optimization",
        "Final training hyperparameter selection",
        "Metadata/export reporting",
    ],
    "optuna_integration_status": OPTUNA_RESULTS.get("status", "disabled"),
    "search_space_summary": {
        "model": [
            "sequence_length", "dense_width", "lstm_hidden_size", "lstm_layers",
            "transformer_hidden_size", "transformer_heads", "transformer_layers",
            "transformer_feedforward_multiplier", "transformer_feedforward_dim", "dropout", "learning_rate", "batch_size", "weight_decay",
        ],
        "forecasting": [
            "recursive_horizon_cutoff", "direct_recursive_blend", "return_clip_threshold",
            "rolling_window_length", "use_volatility_normalization", "scheduled_sampling_end_probability",
            "scheduled_sampling_aux_weight",
        ],
        "ensemble": ["Dense weight", "LSTM weight", "Transformer weight", "horizon_weight_tilt"],
    },
    "validation_methodology_used": OPTUNA_RESULTS.get("validation_methodology"),
    "leakage_validation_status": "preserved: windows end at t-1, scalers fit on training history, labels excluded from features, validation is future tail after embargo",
    "best_trial_summary": {
        "trial_number": OPTUNA_RESULTS.get("best_trial_number"),
        "best_params": OPTUNA_RESULTS.get("best_params"),
        "horizon_metrics": OPTUNA_RESULTS.get("horizon_metrics"),
        "ensemble_weights": OPTUNA_RESULTS.get("ensemble_weights", ensemble_weights),
        "recursive_direct_mix": OPTUNA_RESULTS.get("recursive_direct_mix", metadata.get("forecasting_strategy", {})) if "metadata" in globals() else None,
    },
    "best_forecasting_score": OPTUNA_RESULTS.get("best_score"),
    "checkpoint_compatibility_status": "preserved: Dense.pt/LSTM.pt/Transformer.pt names unchanged; output shape remains 3 quantiles x 5 horizons; metadata records seq_len/features/model capacity",
    "optimized_5_day_pipeline_passed_validation": bool(trained_models) and len(test_metrics_df) == N_HORIZONS,
}
print(json.dumps(summary_payload, indent=2, default=str))


{
  "sections_modified": [
    "Imports/configuration",
    "Model architecture capacity parameters",
    "Training utility pruning/weight decay support",
    "Optuna walk-forward hyperparameter optimization",
    "Final training hyperparameter selection",
    "Metadata/export reporting"
  ],
  "optuna_integration_status": "disabled",
  "search_space_summary": {
    "model": [
      "sequence_length",
      "dense_width",
      "lstm_hidden_size",
      "lstm_layers",
      "transformer_hidden_size",
      "transformer_heads",
      "transformer_layers",
      "transformer_feedforward_multiplier",
      "transformer_feedforward_dim",
      "dropout",
      "learning_rate",
      "batch_size",
      "weight_decay"
    ],
    "forecasting": [
      "recursive_horizon_cutoff",
      "direct_recursive_blend",
      "return_clip_threshold",
      "rolling_window_length",
      "use_volatility_normalization",
      "scheduled_sampling_end_probability",
      "scheduled_sampling_aux_weight"
 